# AKI at ED admission — integrated final submission

**Primary:** observed creatinine AKI Stage ≥1. **Secondary:** Stage ≥2 in the same labelled population.

This notebook extracts raw MIMIC data from Google BigQuery on each run, constructs the cohort and labels,
fits preprocessing and 14 models from scratch, and evaluates both outcomes. It has **no local input-data,
pretrained-model, or teammate-script dependency**. Existing CSV/NPY/NPZ/joblib files are never read.
All analysis code is embedded below. New outputs use a unique directory relative to the working directory.

| Contributor | Integrated work |
|---|---|
| Person 1 | BigQuery extraction, eligible cohort, Decimal-based creatinine phenotype, predictor cleaning |
| Person 2 | Independent phenotype QA, descriptive tables, patient split (seed 90089), train-only preprocessing |
| Person 3 | Dummy, age/gender LR, structured LR, structured + chief complaint LR for both outcomes |
| Person 4 | XGBoost search and explicit sparse structural-zero handling; P5 extends the documented training recipe to Stage ≥2 |
| Person 5 | Integration, validation-only thresholds, common test metrics, paired patient Bootstrap, sensitivity analyses and plots |

**Run:** select the existing `sml` Python kernel, set `BILLING_PROJECT` below (or `GOOGLE_CLOUD_PROJECT`),
and use Run All. The Google account supplying cached pandas-gbq credentials or ADC must have permission to
access PhysioNet MIMIC and run BigQuery jobs in that project. Credentials stay outside the notebook.
The notebook reuses pandas-gbq's cached/browser authentication, as used in the team's local environment.
ADC is also supported. No Colab-only authentication is used. Internet access and authorized BigQuery billing are necessary.

The historical test results were already known when this integration was written. This is a reproducibility
rerun of the team's protocol, **not a new external or previously unseen validation study**. Within this run,
test predictions are generated only after the models and validation thresholds are frozen.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal
import copy
import hashlib
import importlib.metadata
import json
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.metrics import precision_recall_curve, roc_curve
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier
from google.cloud import bigquery
import google.auth
import pandas_gbq.auth

BILLING_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()  # set your own authorized billing project
ED = "physionet-data.mimiciv_ed"
HOSP = "physionet-data.mimiciv_2_2_hosp"
SPLIT_SEED, MODEL_SEED, BOOTSTRAP_SEED = 90089, 42, 9008905
SENSITIVITY_TARGET = 0.80
BOOTSTRAP_REPLICATES = 1000
N_JOBS = min(8, os.cpu_count() or 1)
# Bytes are a scan limit, not a currency estimate. Each query is dry-run before execution.
MAX_QUERY_BYTES = 100 * 1024**3
MAX_TOTAL_QUERY_BYTES = 200 * 1024**3
REQUIRE_HISTORICAL_COUNTS = True
OUTPUT_BASE = Path("final_submission_runs")
assert not OUTPUT_BASE.is_absolute(), "Use a relative output directory."
assert Path(sys.prefix).name.lower() == "sml", "Select the sml Python environment."

def utc_now():
    return datetime.now(timezone.utc).isoformat()

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = OUTPUT_BASE / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
TABLES, FIGURES, CHECKS, PRIVATE = [RUN_DIR / p for p in ("tables", "figures", "checks", "private")]
for directory in (TABLES, FIGURES, CHECKS, PRIVATE):
    directory.mkdir()
OUT_DIR = TABLES  # P2's newly generated tables

def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

def save_csv(frame, filename, directory=TABLES):
    frame.to_csv(directory / filename, index=False, encoding="utf-8-sig")

ENVIRONMENT = {name: importlib.metadata.version(name) for name in (
    "numpy", "pandas", "scipy", "scikit-learn", "xgboost", "matplotlib", "google-cloud-bigquery", "pandas-gbq")}
ENVIRONMENT.update(python=sys.version.split()[0], environment=Path(sys.prefix).name)
write_json(CHECKS / "environment.json", ENVIRONMENT)
write_json(CHECKS / "run_status.json", {"status": "started", "started_at_utc": utc_now(),
    "raw_source": "Google BigQuery", "reads_existing_data_or_models": False})
np.random.seed(SPLIT_SEED)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 10})
print("Run directory:", RUN_DIR)
print("Environment:", ENVIRONMENT)

## Shared definitions and checks before data access

P2 preprocessing is fitted on training patients only. Numeric missingness indicators are used for LR;
XGBoost retains numeric NaN. Clinical text uses training-only TF-IDF unigrams/bigrams (`min_df=5`).
Feature dimensions are derived from the fresh training data rather than hard-coded to the previous handoff.

In [ ]:
"""Shared preprocessing for the MIMIC-IV-ED AKI project.

Every transformer here must be fitted on the TRAINING partition only, then
applied with .transform() to validation and test.

Keep this file next to the saved .joblib preprocessors. Unpickling them
imports this module.
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Model inputs (identical to aki_handoff/predictor_columns.csv)
PREDICTORS = [
    "age", "gender", "arrival_transport",
    "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp",
    "acuity", "pain_numeric", "pain_status", "chiefcomplaint",
]

NUMERIC = ["age", "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp",
           "acuity", "pain_numeric"]
# Numeric fields with missing values; each gets a 0/1 missingness indicator
MISSING_FLAGGED = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp",
                   "acuity", "pain_numeric"]
CATEGORICAL = ["gender", "arrival_transport", "pain_status"]
TEXT = "chiefcomplaint"

# Keeps clinical shorthand such as "s/p", "n/v" and "r/o" as single tokens
TOKEN_PATTERN = r"[a-z0-9]+(?:/[a-z0-9]+)*"


def clean_text(x):
    """Lowercase, trim and fill missing chief complaints with empty text."""
    return pd.Series(x).fillna("").astype(str).str.lower().str.strip().to_numpy()


def _categorical():
    return Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])


def _text():
    return Pipeline([
        ("clean", FunctionTransformer(clean_text, feature_names_out="one-to-one")),
        ("tfidf", TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2),
                                  min_df=5, sublinear_tf=True)),
    ])


def build_lr_preprocessor(include_text=False):
    """For logistic regression: median imputation + missingness indicators +
    standard scaling of numeric fields, one-hot categoricals, optional TF-IDF."""
    parts = [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("miss", MissingIndicator(features="all"), MISSING_FLAGGED),
        ("cat", _categorical(), CATEGORICAL),
    ]
    if include_text:
        parts.append(("text", _text(), TEXT))
    return ColumnTransformer(parts, remainder="drop", sparse_threshold=0.3 if include_text else 0)


def build_tree_preprocessor(include_text=False):
    """For tree models (e.g. XGBoost): numeric fields passed through with NaN
    kept (XGBoost learns a missing-value direction), one-hot categoricals,
    optional TF-IDF. No scaling or imputation."""
    parts = [
        ("num", "passthrough", NUMERIC),
        ("cat", _categorical(), CATEGORICAL),
    ]
    if include_text:
        parts.append(("text", _text(), TEXT))
    return ColumnTransformer(parts, remainder="drop", sparse_threshold=0.3 if include_text else 0)


PREPROCESSORS = {
    "lr_structured": lambda: build_lr_preprocessor(include_text=False),
    "lr_text": lambda: build_lr_preprocessor(include_text=True),
    "tree_structured": lambda: build_tree_preprocessor(include_text=False),
    "tree_text": lambda: build_tree_preprocessor(include_text=True),
}

In [ ]:
"""Frozen-threshold metrics and patient-cluster weighted bootstrap primitives."""
from __future__ import annotations

import numpy as np
from scipy.optimize import brentq
from scipy.special import expit, logit

METRIC_NAMES = [
    "average_precision", "auroc", "brier_score", "sensitivity", "specificity", "ppv", "npv",
    "false_positive_rate", "flagged_fraction", "false_positives_per_1000",
    "calibration_intercept", "calibration_slope", "calibration_in_the_large",
]


def divide(a, b):
    return float(a / b) if b else float("nan")


def choose_threshold(y, p, sensitivity_target):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    if not 0 < sensitivity_target <= 1 or y.sum() == 0:
        raise ValueError("A valid sensitivity target and positive observations are required")
    # kth largest positive probability is the largest attainable qualifying cutoff;
    # every tied score is included by the >= decision rule.
    positive_probabilities = np.sort(p[y == 1])[::-1]
    count_needed = int(np.ceil(sensitivity_target * len(positive_probabilities)))
    return float(positive_probabilities[count_needed - 1])


def calibration(y, p, weights, initial=None):
    """Joint logit calibration intercept/slope plus offset-only calibration-in-large.

    These are diagnostic fits only; they never alter prediction probabilities.
    A constant score has no identifiable joint intercept/slope, but has a CITL.
    """
    x = logit(np.clip(p, 1e-12, 1 - 1e-12))
    n = weights.sum()
    positives = np.dot(weights, y)
    if positives <= 0 or positives >= n:
        return np.nan, np.nan, np.nan
    citl = brentq(lambda a: np.dot(weights, expit(x + a)) - positives, -60.0, 60.0, xtol=1e-10)
    if np.ptp(x[weights > 0]) < 1e-12:
        return np.nan, np.nan, float(citl)
    a, b = initial if initial is not None and np.isfinite(initial).all() else (citl, 1.0)
    converged = False
    for _ in range(60):
        eta = a + b * x
        fitted = expit(eta)
        residual = weights * (fitted - y)
        curvature = weights * fitted * (1 - fitted)
        gradient = np.array([residual.sum(), np.dot(residual, x)])
        h00, h01, h11 = curvature.sum(), np.dot(curvature, x), np.dot(curvature, x * x)
        hessian = np.array([[h00, h01], [h01, h11]])
        if np.max(np.abs(gradient)) < 1e-8:
            converged = True
            break
        try:
            step = np.linalg.solve(hessian, gradient)
        except np.linalg.LinAlgError:
            break
        loss = np.dot(weights, np.logaddexp(0, eta) - y * eta)
        scale = 1.0
        for _ in range(25):
            new_a, new_b = a - scale * step[0], b - scale * step[1]
            new_eta = new_a + new_b * x
            new_loss = np.dot(weights, np.logaddexp(0, new_eta) - y * new_eta)
            if new_loss <= loss + 1e-10:
                break
            scale *= 0.5
        a, b = float(new_a), float(new_b)
        if np.max(np.abs(scale * step)) < 1e-8:
            converged = True
            break
    return (a, b, float(citl)) if converged else (np.nan, np.nan, float(citl))


class PreparedMetrics:
    """Pre-sort scores once; integer cluster weights exactly represent row replication."""

    def __init__(self, y, p, threshold):
        self.y = np.asarray(y, dtype=float)
        self.p = np.asarray(p, dtype=float)
        if self.y.ndim != 1 or self.y.shape != self.p.shape or not np.isin(self.y, [0, 1]).all():
            raise ValueError("Invalid outcomes/prediction dimensions")
        if not np.isfinite(self.p).all() or not ((self.p >= 0) & (self.p <= 1)).all():
            raise ValueError("Invalid probabilities")
        self.threshold = float(threshold)
        self.flag = self.p >= self.threshold
        self.order = np.argsort(-self.p, kind="stable")
        sorted_p = self.p[self.order]
        self.ends = np.r_[np.flatnonzero(sorted_p[:-1] != sorted_p[1:]), len(sorted_p) - 1]
        self.squared_error = (self.y - self.p) ** 2
        self.calibration_initial = None

    def evaluate(self, weights=None, include_calibration=True):
        w = np.ones(len(self.y)) if weights is None else np.asarray(weights, dtype=float)
        if w.shape != self.y.shape or (w < 0).any() or w.sum() <= 0:
            raise ValueError("Invalid evaluation weights")
        y, p = self.y, self.p
        positive = np.dot(w, y)
        total = w.sum()
        negative = total - positive
        sorted_positive = (w * y)[self.order]
        sorted_negative = (w * (1 - y))[self.order]
        tps = np.cumsum(sorted_positive)[self.ends]
        fps = np.cumsum(sorted_negative)[self.ends]
        precision = np.divide(tps, tps + fps, out=np.zeros_like(tps), where=tps + fps > 0)
        ap = float(np.dot(np.diff(np.r_[0.0, tps]) / positive, precision)) if positive else np.nan
        auc = float(np.dot(np.diff(np.r_[0.0, fps]), (tps + np.r_[0.0, tps[:-1]]) / 2) / (positive * negative)) if positive and negative else np.nan
        tp = np.dot(w, y * self.flag)
        fp = np.dot(w, (1 - y) * self.flag)
        fn = positive - tp
        tn = negative - fp
        result = {
            "average_precision": ap, "auroc": auc, "brier_score": float(np.dot(w, self.squared_error) / total),
            "sensitivity": divide(tp, positive), "specificity": divide(tn, negative),
            "ppv": divide(tp, tp + fp), "npv": divide(tn, tn + fn),
            "false_positive_rate": divide(fp, negative), "flagged_fraction": divide(tp + fp, total),
            "false_positives_per_1000": float(1000 * fp / total),
            "true_positives": float(tp), "false_positives": float(fp),
            "true_negatives": float(tn), "false_negatives": float(fn),
        }
        if include_calibration:
            a, b, citl = calibration(y, p, w, self.calibration_initial)
            result.update(calibration_intercept=a, calibration_slope=b, calibration_in_the_large=citl)
            if weights is None:
                self.calibration_initial = (a, b)
        else:
            result.update({name: np.nan for name in METRIC_NAMES[-3:]})
        return result

In [ ]:
def clean_predictors(raw):
    """Apply the agreed section-6 cleaning rules to raw triage fields."""
    out = pd.DataFrame(index=raw.index)
    out["age"] = pd.to_numeric(raw["age"])
    out["gender"] = raw["gender"]
    out["arrival_transport"] = raw["arrival_transport"]

    temp = pd.to_numeric(raw["temperature"], errors="coerce")
    clean_temp = temp.where((temp > 70) & (temp < 120))
    celsius = temp.between(30, 43)
    clean_temp[celsius] = temp[celsius] * 9 / 5 + 32
    out["temperature"] = clean_temp

    upper = {"heartrate": 300, "resprate": 70, "sbp": 400, "dbp": 300}
    for col, limit in upper.items():
        v = pd.to_numeric(raw[col], errors="coerce")
        out[col] = v.where((v > 0) & (v < limit))
    o2 = pd.to_numeric(raw["o2sat"], errors="coerce")
    out["o2sat"] = o2.where((o2 > 0) & (o2 <= 100))

    acuity = pd.to_numeric(raw["acuity"], errors="coerce")
    out["acuity"] = acuity.where(acuity.between(1, 5))

    pain = raw["pain"].astype("string").str.strip()
    numeric = pain.str.fullmatch(r"([0-9]|10)(\.0+)?", na=False)
    out["pain_numeric"] = pd.to_numeric(pain.where(numeric), errors="coerce")
    missing = pain.isna() | pain.eq("").fillna(False)
    out["pain_status"] = "other_recorded"
    out.loc[missing, "pain_status"] = "missing"
    out.loc[numeric, "pain_status"] = "numeric"

    cc = raw["chiefcomplaint"].astype("string").str.strip()
    out["chiefcomplaint"] = cc.replace("", pd.NA)
    return out[[c for c in predictor_columns]]

predictor_columns = PREDICTORS.copy()

In [ ]:
def derive_phenotype(cohort, creatinine_raw):
    """P1 cells 27–66: unchanged computation, in-memory inputs; displays removed."""
    cohort = cohort.copy()
    cohort['endpoint_time'] = cohort['intime'] + pd.Timedelta(hours=24)
    early_discharge = cohort['dischtime'] < cohort['endpoint_time']
    cohort.loc[early_discharge, 'endpoint_time'] = cohort.loc[early_discharge, 'dischtime']
    cohort['followup_hours'] = (cohort['endpoint_time'] - cohort['intime']).dt.total_seconds() / 3600

    creatinine = creatinine_raw[creatinine_raw['itemid'].isin([50912, 52546])].copy()
    creatinine_whole_blood = creatinine_raw[creatinine_raw['itemid'] == 52024].copy()
    creatinine['charttime'] = pd.to_datetime(creatinine['charttime'])
    creatinine_values = pd.to_numeric(creatinine['valuenum'], errors='coerce')
    same_time = creatinine.duplicated(subset=['subject_id', 'charttime'], keep=False)

    valid_value = (creatinine_values > 0) & (creatinine_values <= 150)
    creatinine_valid = creatinine[valid_value].copy()
    values_per_time = creatinine_valid.groupby(['subject_id', 'charttime'])['valuenum'].nunique()

    value_counts = creatinine_valid.groupby(['subject_id', 'charttime'])['valuenum'].transform('nunique')
    creatinine_conflicts = creatinine_valid[value_counts > 1].copy()
    creatinine_clean = creatinine_valid[value_counts == 1].copy()
    sample_times = creatinine_clean[['subject_id', 'charttime']].drop_duplicates()

    from decimal import Decimal
    creatinine_clean['creat_decimal'] = creatinine_clean['valuenum'].astype(str).map(Decimal)
    difference = Decimal('0.9') - Decimal('0.6')

    windows = cohort[['subject_id', 'stay_id', 'hadm_id', 'intime', 'endpoint_time']].copy()
    windows = windows.rename(columns={'hadm_id': 'ed_hadm_id'})
    labs = creatinine_clean[['labevent_id', 'subject_id', 'hadm_id', 'charttime', 'creat_decimal']].copy()
    patient_ids = cohort['subject_id'].unique()
    matched_parts = []
    for start in range(0, len(patient_ids), 5000):
        batch_ids = patient_ids[start:start + 5000]
        batch_labs = labs[labs['subject_id'].isin(batch_ids)]
        batch_windows = windows[windows['subject_id'].isin(batch_ids)]
        matches = batch_labs.merge(batch_windows, on='subject_id')
        within_window = (matches['charttime'] >= matches['intime']) & (matches['charttime'] <= matches['endpoint_time'])
        admission_match = matches['hadm_id'].isna() | (matches['hadm_id'] == matches['ed_hadm_id'])
        matched_parts.append(matches[within_window & admission_match].copy())
    current_matches = pd.concat(matched_parts, ignore_index=True)
    encounters_per_event = current_matches.groupby('labevent_id')['stay_id'].nunique()

    sorted_matches = current_matches.sort_values(['labevent_id', 'intime', 'stay_id'], ascending=[True, False, True])
    current_events = sorted_matches.drop_duplicates(subset='labevent_id', keep='first').copy()
    current_samples = current_events.drop_duplicates(subset=['stay_id', 'charttime']).copy()
    sample_counts = current_samples.groupby('stay_id').size()
    cohort['current_sample_count'] = cohort['stay_id'].map(sample_counts).fillna(0).astype(int)

    sampling = current_samples.groupby('stay_id')['charttime'].agg(['min', 'max'])
    sampling = sampling.rename(columns={'min': 'first_sample_time', 'max': 'last_sample_time'})
    cohort = cohort.join(sampling, on='stay_id')
    cohort['first_sample_delay_hours'] = (cohort['first_sample_time'] - cohort['intime']).dt.total_seconds() / 3600
    cohort['sampling_span_hours'] = (cohort['last_sample_time'] - cohort['first_sample_time']).dt.total_seconds() / 3600
    cohort['endpoint_gap_hours'] = (cohort['endpoint_time'] - cohort['last_sample_time']).dt.total_seconds() / 3600
    serial = cohort[cohort['current_sample_count'] >= 2]

    reference_samples = creatinine_clean[['subject_id', 'charttime', 'creat_decimal']].drop_duplicates(subset=['subject_id', 'charttime']).copy()
    reference_samples = reference_samples.sort_values(['subject_id', 'charttime'])
    reference_samples = reference_samples.rename(columns={'charttime': 'reference_time', 'creat_decimal': 'reference_creatinine'})

    samples = current_samples[['subject_id', 'stay_id', 'charttime', 'creat_decimal']].copy()
    samples = samples.reset_index(drop=True)
    samples['sample_id'] = samples.index
    patient_ids = samples['subject_id'].unique()
    comparison_parts = []
    for start in range(0, len(patient_ids), 1000):
        batch_ids = patient_ids[start:start + 1000]
        batch_samples = samples[samples['subject_id'].isin(batch_ids)]
        batch_history = reference_samples[reference_samples['subject_id'].isin(batch_ids)]
        pairs = batch_samples.merge(batch_history, on='subject_id')
        time_gap = pairs['charttime'] - pairs['reference_time']
        earlier_7d = (time_gap > pd.Timedelta(0)) & (time_gap <= pd.Timedelta(days=7))
        pairs = pairs[earlier_7d].copy()
        minimum_7d = pairs.groupby('sample_id')['reference_creatinine'].min()
        within_48h = pairs['charttime'] - pairs['reference_time'] <= pd.Timedelta(hours=48)
        minimum_48h = pairs[within_48h].groupby('sample_id')['reference_creatinine'].min()
        batch_samples = batch_samples.copy()
        batch_samples['minimum_48h'] = batch_samples['sample_id'].map(minimum_48h)
        batch_samples['minimum_7d'] = batch_samples['sample_id'].map(minimum_7d)
        comparison_parts.append(batch_samples)
    sample_comparisons = pd.concat(comparison_parts, ignore_index=True)

    sample_results = sample_comparisons.copy()
    sample_results['rise_48h'] = False
    sample_results['rise_7d'] = False
    has_48h = sample_results['minimum_48h'].notna()
    sample_results.loc[has_48h, 'rise_48h'] = sample_results.loc[has_48h, 'creat_decimal'] - sample_results.loc[has_48h, 'minimum_48h'] >= Decimal('0.3')
    has_7d = sample_results['minimum_7d'].notna()
    sample_results.loc[has_7d, 'rise_7d'] = sample_results.loc[has_7d, 'creat_decimal'] >= sample_results.loc[has_7d, 'minimum_7d'] * Decimal('1.5')
    sample_results['positive_sample'] = sample_results['rise_48h'] | sample_results['rise_7d']
    sample_results['has_comparator'] = has_7d
    positive_samples = sample_results[sample_results['positive_sample']]

    positive_stays = positive_samples['stay_id'].unique()
    cohort['observed_positive'] = cohort['stay_id'].isin(positive_stays)
    first_positive_time = positive_samples.groupby('stay_id')['charttime'].min()
    cohort['first_positive_time'] = cohort['stay_id'].map(first_positive_time)
    cohort['hours_to_first_positive'] = (cohort['first_positive_time'] - cohort['intime']).dt.total_seconds() / 3600
    bad_positive_times = cohort['observed_positive'] & ((cohort['first_positive_time'] < cohort['intime']) | (cohort['first_positive_time'] > cohort['endpoint_time']))

    first_samples = sample_results.sort_values('charttime').drop_duplicates(subset='stay_id').copy()
    first_samples['first_sample_status'] = 'unassessable'
    assessable_no_rise = first_samples['has_comparator'] & ~first_samples['positive_sample']
    first_samples.loc[assessable_no_rise, 'first_sample_status'] = 'no qualifying rise'
    first_samples.loc[first_samples['positive_sample'], 'first_sample_status'] = 'positive'
    first_status = first_samples.set_index('stay_id')['first_sample_status']
    cohort['first_sample_status'] = cohort['stay_id'].map(first_status)
    cohort['first_sample_status'] = cohort['first_sample_status'].fillna('no current sample')

    sample_results['creatinine_stage'] = pd.Series(pd.NA, index=sample_results.index, dtype='Int64')
    sample_results.loc[sample_results['positive_sample'], 'creatinine_stage'] = 1
    has_reference = sample_results['minimum_7d'].notna()
    current = sample_results.loc[has_reference, 'creat_decimal']
    reference = sample_results.loc[has_reference, 'minimum_7d']
    stage2_index = current[current >= reference * Decimal('2')].index
    stage3_index = current[current >= reference * Decimal('3')].index
    sample_results.loc[stage2_index, 'creatinine_stage'] = 2
    sample_results.loc[stage3_index, 'creatinine_stage'] = 3
    absolute_stage3 = sample_results['positive_sample'] & (sample_results['creat_decimal'] >= Decimal('4'))
    sample_results.loc[absolute_stage3, 'creatinine_stage'] = 3
    highest_stage = sample_results.groupby('stay_id')['creatinine_stage'].max()
    cohort['highest_creatinine_stage'] = cohort['stay_id'].map(highest_stage)

    severe_samples = sample_results[sample_results['creatinine_stage'] >= 2].copy()
    first_stage2_time = severe_samples.groupby('stay_id')['charttime'].min()
    cohort['first_stage2_time'] = cohort['stay_id'].map(first_stage2_time)
    cohort['observed_stage2_positive'] = cohort['first_stage2_time'].notna()
    invalid_secondary = cohort['observed_stage2_positive'] & ~cohort['observed_positive']
    invalid_order = cohort['first_stage2_time'] < cohort['first_positive_time']

    has_comparison = sample_results.groupby('stay_id')['has_comparator'].any()
    cohort['has_timed_comparison'] = cohort['stay_id'].map(has_comparison).eq(True)
    cohort['serial_no_rise'] = (cohort['current_sample_count'] >= 2) & cohort['has_timed_comparison'] & ~cohort['observed_positive']

    cohort['label_state'] = 'indeterminate'
    cohort.loc[cohort['current_sample_count'] == 0, 'label_state'] = 'no_usable_current_creatinine'
    cohort.loc[cohort['serial_no_rise'], 'label_state'] = 'negative'
    cohort.loc[cohort['observed_positive'], 'label_state'] = 'positive'
    cohort['aki_label'] = pd.Series(pd.NA, index=cohort.index, dtype='Int64')
    cohort.loc[cohort['label_state'] == 'negative', 'aki_label'] = 0
    cohort.loc[cohort['label_state'] == 'positive', 'aki_label'] = 1

    positive = cohort['label_state'] == 'positive'
    negative = cohort['label_state'] == 'negative'
    unresolved = cohort['label_state'].isin(['indeterminate', 'no_usable_current_creatinine'])
    invalid_negatives = negative & ((cohort['current_sample_count'] < 2) | ~cohort['has_timed_comparison'] | cohort['observed_positive'])
    labelled = cohort[cohort['aki_label'].notna()].copy()
    return cohort, sample_results

In [ ]:
CURRENT_ITEMS = [50912, 52546]

def derive_outcomes(encounters, labs):
    """Re-expression of the notebook's outcome pipeline (cells 27-64)."""
    cohort = encounters.copy()

    # Observation window: earlier of 24 h after arrival and hospital discharge
    cohort["endpoint_time"] = cohort["intime"] + pd.Timedelta(hours=24)
    early = cohort["dischtime"] < cohort["endpoint_time"]
    cohort.loc[early, "endpoint_time"] = cohort.loc[early, "dischtime"]

    # Primary items, value screen, conflicting same-time values removed
    creat = labs[labs["itemid"].isin(CURRENT_ITEMS)].copy()
    v = pd.to_numeric(creat["valuenum"], errors="coerce")
    creat = creat[(v > 0) & (v <= 150)].copy()
    nvals = creat.groupby(["subject_id", "charttime"])["valuenum"].transform("nunique")
    creat = creat[nvals == 1].copy()
    creat["creat_decimal"] = creat["valuenum"].astype(str).map(Decimal)

    # Link samples to windows (matching or missing lab admission ID)
    windows = cohort[["subject_id", "stay_id", "hadm_id", "intime", "endpoint_time"]].rename(
        columns={"hadm_id": "ed_hadm_id"})
    m = creat[["labevent_id", "subject_id", "hadm_id", "charttime", "creat_decimal"]].merge(
        windows, on="subject_id")
    m = m[(m["charttime"] >= m["intime"]) & (m["charttime"] <= m["endpoint_time"])
          & (m["hadm_id"].isna() | (m["hadm_id"] == m["ed_hadm_id"]))]

    # One encounter per lab event: most recent arrival, then smaller stay_id
    ev = m.sort_values(["labevent_id", "intime", "stay_id"], ascending=[True, False, True]) \
          .drop_duplicates("labevent_id")
    cs = ev.drop_duplicates(["stay_id", "charttime"]).copy()
    cohort["current_sample_count"] = cohort["stay_id"].map(cs.groupby("stay_id").size()).fillna(0).astype(int)

    # Strictly earlier references within 48 h / 7 days (any of the patient's usable samples)
    ref = creat[["subject_id", "charttime", "creat_decimal"]].drop_duplicates(["subject_id", "charttime"]) \
        .rename(columns={"charttime": "reference_time", "creat_decimal": "reference_creatinine"})
    s = cs[["subject_id", "stay_id", "charttime", "creat_decimal"]].reset_index(drop=True)
    s["sample_id"] = s.index
    p = s.merge(ref, on="subject_id")
    gap = p["charttime"] - p["reference_time"]
    p = p[(gap > pd.Timedelta(0)) & (gap <= pd.Timedelta(days=7))]
    min7 = p.groupby("sample_id")["reference_creatinine"].min()
    min48 = p[(p["charttime"] - p["reference_time"]) <= pd.Timedelta(hours=48)] \
        .groupby("sample_id")["reference_creatinine"].min()
    s["minimum_48h"] = s["sample_id"].map(min48)
    s["minimum_7d"] = s["sample_id"].map(min7)

    # Qualifying rises (Decimal arithmetic)
    s["rise_48h"] = False
    s["rise_7d"] = False
    h48 = s["minimum_48h"].notna()
    s.loc[h48, "rise_48h"] = (s.loc[h48, "creat_decimal"] - s.loc[h48, "minimum_48h"]) >= Decimal("0.3")
    h7 = s["minimum_7d"].notna()
    s.loc[h7, "rise_7d"] = s.loc[h7, "creat_decimal"] >= s.loc[h7, "minimum_7d"] * Decimal("1.5")
    s["positive_sample"] = s["rise_48h"] | s["rise_7d"]
    s["has_comparator"] = h7

    # Stages
    s["stage"] = pd.Series(pd.NA, index=s.index, dtype="Int64")
    s.loc[s["positive_sample"], "stage"] = 1
    cur, refv = s.loc[h7, "creat_decimal"], s.loc[h7, "minimum_7d"]
    s.loc[cur[cur >= refv * Decimal("2")].index, "stage"] = 2
    s.loc[cur[cur >= refv * Decimal("3")].index, "stage"] = 3
    s.loc[s["positive_sample"] & (s["creat_decimal"] >= Decimal("4")), "stage"] = 3

    # Encounter-level results
    pos_stays = s.loc[s["positive_sample"], "stay_id"].unique()
    cohort["observed_positive"] = cohort["stay_id"].isin(pos_stays)
    cohort["highest_stage"] = cohort["stay_id"].map(s.groupby("stay_id")["stage"].max())
    cohort["stage2_label"] = (cohort["highest_stage"] >= 2).fillna(False).astype(int)

    first = s.sort_values("charttime").drop_duplicates("stay_id").copy()
    first["status"] = "unassessable"
    first.loc[first["has_comparator"] & ~first["positive_sample"], "status"] = "no qualifying rise"
    first.loc[first["positive_sample"], "status"] = "positive"
    cohort["first_sample_status"] = cohort["stay_id"].map(first.set_index("stay_id")["status"]) \
        .fillna("no current sample")

    has_cmp = cohort["stay_id"].map(s.groupby("stay_id")["has_comparator"].any()).eq(True)
    serial_no_rise = (cohort["current_sample_count"] >= 2) & has_cmp & ~cohort["observed_positive"]
    cohort["label_state"] = "indeterminate"
    cohort.loc[cohort["current_sample_count"] == 0, "label_state"] = "no_usable_current_creatinine"
    cohort.loc[serial_no_rise, "label_state"] = "negative"
    cohort.loc[cohort["observed_positive"], "label_state"] = "positive"
    return cohort

In [ ]:
T0 = pd.Timestamp("2150-01-01 08:00")
H = lambda h: pd.Timedelta(hours=h)
M = lambda mins: pd.Timedelta(minutes=mins)

encounters, labs, expected = [], [], []
lab_counter = [0]

def case(cid, description, stays, samples, expect):
    """stays: list of (stay_id, hadm_id, arrival offset, discharge offset).
    samples: list of (offset from T0, value, lab hadm_id or None, itemid).
    expect: {stay_id: (label_state, current_sample_count, highest_stage, first_sample_status)}."""
    subject = 1000 + cid
    for stay_id, hadm, arr, dis in stays:
        encounters.append({"case": cid, "subject_id": subject, "stay_id": stay_id, "hadm_id": hadm,
                           "intime": T0 + arr, "dischtime": T0 + dis})
    for offset, value, hadm, item in samples:
        lab_counter[0] += 1
        labs.append({"labevent_id": lab_counter[0], "subject_id": subject, "hadm_id": hadm,
                     "itemid": item, "charttime": T0 + offset, "valuenum": value})
    for stay_id, (state, n, stage, status) in expect.items():
        expected.append({"case": cid, "description": description, "stay_id": stay_id,
                         "exp_label_state": state, "exp_current_samples": n,
                         "exp_highest_stage": stage, "exp_first_sample_status": status})

CR = 50912
std = lambda sid: [(sid, sid, H(0), H(72))]  # one encounter; hadm_id = stay_id; discharge at 72 h

case(1, "Absolute rise of exactly 0.3 (2.0->2.3) that float arithmetic misses",
     std(101), [(H(1), 2.0, 101, CR), (H(5), 2.3, 101, CR)],
     {101: ("positive", 2, 1, "unassessable")})
case(2, "Rise of 0.29 (2.0->2.29) does not qualify",
     std(102), [(H(1), 2.0, 102, CR), (H(5), 2.29, 102, CR)],
     {102: ("negative", 2, None, "unassessable")})
case(3, "Reference exactly 48 h earlier (pre-arrival, lab hadm missing) is included",
     std(103), [(H(-46), 1.0, None, CR), (H(2), 1.3, 103, CR)],
     {103: ("positive", 1, 1, "positive")})
case(4, "Reference 48 h + 1 min earlier is excluded from the 48 h rule; ratio 1.3 < 1.5",
     std(104), [(H(-46) - M(1), 1.0, None, CR), (H(2), 1.3, 104, CR)],
     {104: ("indeterminate", 1, None, "no qualifying rise")})
case(5, "Reference exactly 7 days earlier with ratio exactly 1.5 qualifies",
     std(105), [(H(1) - pd.Timedelta(days=7), 1.0, None, CR), (H(1), 1.5, 105, CR)],
     {105: ("positive", 1, 1, "positive")})
case(6, "Reference 7 days + 1 min earlier is not a comparator",
     std(106), [(H(1) - pd.Timedelta(days=7) - M(1), 1.0, None, CR), (H(1), 1.5, 106, CR)],
     {106: ("indeterminate", 1, None, "unassessable")})
case(7, "Agreeing duplicate records at one time count once; same-time record is not a comparator",
     std(107), [(H(1), 1.0, 107, CR), (H(1), 1.0, 107, CR), (H(3), 1.0, 107, CR)],
     {107: ("negative", 2, None, "unassessable")})
case(8, "Conflicting values at one time are removed (as current samples and as references)",
     std(108), [(H(1), 1.0, 108, CR), (H(1), 2.0, 108, CR), (H(3), 1.0, 108, CR)],
     {108: ("indeterminate", 1, None, "unassessable")})
case(9, "Samples exactly at arrival and exactly at 24 h are both in the window",
     std(109), [(H(0), 1.0, 109, CR), (H(24), 1.5, 109, CR)],
     {109: ("positive", 2, 1, "unassessable")})
case(10, "Sample 1 min after 24 h is outside the window",
     std(110), [(H(0), 1.0, 110, CR), (H(24) + M(1), 1.5, 110, CR)],
     {110: ("indeterminate", 1, None, "unassessable")})
case(11, "Hospital discharge before 24 h shortens the window",
     [(111, 111, H(0), H(10))], [(H(1), 1.0, 111, CR), (H(12), 2.0, 111, CR)],
     {111: ("indeterminate", 1, None, "unassessable")})
case(12, "Lab from another admission is not a current sample but is a valid reference",
     std(112), [(H(2), 1.0, 999, CR), (H(6), 1.4, 112, CR)],
     {112: ("positive", 1, 1, "positive")})
case(13, "Overlapping encounters: shared sample goes to the most recent arrival",
     [(113, 1131, H(0), H(72)), (114, 1132, H(5), H(72))],
     [(H(2), 1.0, None, CR), (H(6), 1.4, None, CR), (H(8), 1.0, 1131, CR)],
     {113: ("negative", 2, None, "unassessable"), 114: ("positive", 1, 1, "positive")})
case(14, "Identical arrival times: shared samples go to the smaller stay_id",
     [(116, 1161, H(0), H(72)), (115, 1151, H(0), H(72))],
     [(H(1), 1.0, None, CR), (H(3), 1.5, None, CR)],
     {115: ("positive", 2, 1, "unassessable"), 116: ("no_usable_current_creatinine", 0, None, "no current sample")})
case(15, "Ratio exactly 2.0 gives stage 2",
     std(117), [(H(1), 1.0, 117, CR), (H(9), 2.0, 117, CR)],
     {117: ("positive", 2, 2, "unassessable")})
case(16, "Ratio exactly 3.0 gives stage 3",
     std(118), [(H(1), 1.0, 118, CR), (H(9), 3.0, 118, CR)],
     {118: ("positive", 2, 3, "unassessable")})
case(17, "Creatinine >= 4 with a qualifying 0.3 rise (3.7->4.0) gives stage 3",
     std(119), [(H(1), 3.7, 119, CR), (H(9), 4.0, 119, CR)],
     {119: ("positive", 2, 3, "unassessable")})
case(18, "Creatinine >= 4 without a qualifying rise (4.0->4.1) stays negative",
     std(120), [(H(1), 4.0, 120, CR), (H(9), 4.1, 120, CR)],
     {120: ("negative", 2, None, "unassessable")})
case(19, "Zero creatinine is screened out",
     std(121), [(H(1), 0.0, 121, CR), (H(3), 1.0, 121, CR)],
     {121: ("indeterminate", 1, None, "unassessable")})
case(20, "Whole-blood item 52024 is ignored for the primary endpoint",
     std(122), [(H(1), 1.0, 122, 52024), (H(3), 2.0, 122, CR)],
     {122: ("indeterminate", 1, None, "unassessable")})
case(21, "Item 52546 counts as a current sample",
     std(123), [(H(1), 1.0, 123, 52546), (H(3), 2.0, 123, CR)],
     {123: ("positive", 2, 2, "unassessable")})
case(22, "No creatinine in the window",
     std(124), [(H(-30), 1.0, None, CR)],
     {124: ("no_usable_current_creatinine", 0, None, "no current sample")})
case(23, "Already positive at first sample vs pre-arrival reference, later value falls",
     std(125), [(H(-24), 1.0, None, CR), (H(1), 1.6, 125, CR), (H(10), 1.2, 125, CR)],
     {125: ("positive", 2, 1, "positive")})
case(24, "Minimum (not most recent) reference is used",
     std(126), [(H(-30), 1.0, None, CR), (H(-2), 1.4, None, CR), (H(1), 1.35, 126, CR), (H(8), 1.36, 126, CR)],
     {126: ("positive", 2, 1, "positive")})

enc_df = pd.DataFrame(encounters)
lab_df = pd.DataFrame(labs)
lab_df["hadm_id"] = lab_df["hadm_id"].astype("Int64")
enc_df["hadm_id"] = enc_df["hadm_id"].astype("Int64")

result = derive_outcomes(enc_df.drop(columns="case"), lab_df)
tests = pd.DataFrame(expected).merge(
    result[["stay_id", "label_state", "current_sample_count", "highest_stage", "first_sample_status"]],
    on="stay_id")
tests["exp_highest_stage"] = tests["exp_highest_stage"].astype("Int64")
tests["pass"] = (
    (tests["label_state"] == tests["exp_label_state"])
    & (tests["current_sample_count"] == tests["exp_current_samples"])
    & ((tests["highest_stage"] == tests["exp_highest_stage"]).fillna(False)
       | (tests["highest_stage"].isna() & tests["exp_highest_stage"].isna()))
    & (tests["first_sample_status"] == tests["exp_first_sample_status"])
)
tests.to_csv(OUT_DIR / "qa_synthetic_boundary_tests.csv", index=False, encoding="utf-8-sig")
print(f"Synthetic cases passing: {tests['pass'].sum()} / {len(tests)}")
tests[["case", "description", "stay_id", "exp_label_state", "label_state",
       "exp_current_samples", "current_sample_count", "exp_highest_stage", "highest_stage", "pass"]]
assert tests["pass"].all(), "P2 synthetic boundary tests failed."
p1_synthetic, _ = derive_phenotype(enc_df.drop(columns="case"), lab_df)
p1_check = p1_synthetic.set_index("stay_id").sort_index()
p2_check = result.set_index("stay_id").sort_index()
for column in ("label_state", "current_sample_count", "first_sample_status"):
    assert p1_check[column].equals(p2_check[column]), column
assert p1_check["highest_creatinine_stage"].equals(p2_check["highest_stage"])
assert np.array_equal(p1_check["observed_stage2_positive"].astype(int), p2_check["stage2_label"])
write_json(CHECKS / "phenotype_boundary_tests.json", {"cases": 24, "encounter_assertions": len(tests),
    "p2_reference_passed": True, "production_p1_matches_independent_p2": True})
print("P1 and independent P2 agree on all 26 synthetic encounters (24 boundary cases).")

## Part 1 — fresh extraction and cohort construction (Person 1)

Only the following raw BigQuery tables are read: ED `edstays`, `triage`; HOSP `patients`, `admissions`,
`diagnoses_icd`, `d_labitems`, `labevents`. A successful run logs table metadata, SQL hashes, scan sizes
and query job IDs. Query-result caching is disabled. An inaccessible source stops the run; there is no
fallback to existing files. `triage` is the source-table name; the study prediction anchor is **at ED admission**.

In [ ]:
if not BILLING_PROJECT:
    write_json(CHECKS / "run_status.json", {"status": "blocked_before_extraction",
        "reason": "BILLING_PROJECT is empty; authorized BigQuery project and ADC are required.",
        "updated_at_utc": utc_now()})
    raise RuntimeError("Set BILLING_PROJECT or GOOGLE_CLOUD_PROJECT and configure authorized Google ADC; no existing CSV will be used.")
# Same cached/browser OAuth mechanism as pandas_gbq.read_gbq. No credential contents are displayed.
credentials, _ = pandas_gbq.auth.get_credentials(
    project_id=BILLING_PROJECT, reauth=True, auth_local_webserver=True
)
client = bigquery.Client(project=BILLING_PROJECT, credentials=credentials)
query_audit, table_audit = [], []
raw_tables = [f"{ED}.{name}" for name in ("edstays", "triage")] + [
    f"{HOSP}.{name}" for name in ("patients", "admissions", "diagnoses_icd", "d_labitems", "labevents")]
for table_id in raw_tables:
    table = client.get_table(table_id)
    table_audit.append({"table": table_id, "modified": str(table.modified), "etag": table.etag,
        "num_rows": table.num_rows, "schema": [(f.name, f.field_type, f.mode) for f in table.schema]})
write_json(CHECKS / "source_tables.json", table_audit)

def query_dataframe(query, query_parameters=None):
    parameters = query_parameters or []
    dry_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False, query_parameters=parameters)
    dry_job = client.query(query, job_config=dry_config)
    estimated = dry_job.total_bytes_processed
    if estimated is None:
        raise RuntimeError("BigQuery returned no scan estimate; query not executed.")
    estimated = int(estimated)
    if estimated > MAX_QUERY_BYTES or sum(q["estimated_bytes"] for q in query_audit) + estimated > MAX_TOTAL_QUERY_BYTES:
        raise RuntimeError(f"Scan limit exceeded ({estimated / 1024**3:.2f} GiB for next query); review limits before rerunning.")
    config = bigquery.QueryJobConfig(use_query_cache=False, query_parameters=parameters,
                                    maximum_bytes_billed=MAX_QUERY_BYTES)
    job = client.query(query, job_config=config)
    frame = job.result().to_dataframe(create_bqstorage_client=False, int_dtype=None, bool_dtype=None)
    query_audit.append({"query_number": len(query_audit) + 1, "sql": query,
        "sql_sha256": hashlib.sha256(query.encode()).hexdigest(), "job_id": job.job_id,
        "estimated_bytes": estimated, "processed_bytes": job.total_bytes_processed,
        "billed_bytes": job.total_bytes_billed, "cache_hit": job.cache_hit,
        "rows_downloaded": len(frame), "completed_at_utc": utc_now(),
        "array_parameter_lengths": {p.name: len(p.values) for p in parameters}})
    write_json(CHECKS / "query_audit.json", query_audit)
    print(f"Raw query {len(query_audit)}: {len(frame):,} rows; scan {estimated / 1024**3:.3f} GiB")
    return frame

def patient_parameter(ids):
    return [bigquery.ArrayQueryParameter("patient_ids", "INT64", [int(i) for i in ids])]

In [ ]:
edstays_raw = query_dataframe(f"SELECT * FROM `{ED}.edstays`")
ed_patient_ids = edstays_raw["subject_id"].dropna().unique().tolist()
patients_raw = query_dataframe(f"SELECT * FROM `{HOSP}.patients` WHERE subject_id IN UNNEST(@patient_ids)", patient_parameter(ed_patient_ids))
admissions_raw = query_dataframe(f"SELECT * FROM `{HOSP}.admissions` WHERE subject_id IN UNNEST(@patient_ids)", patient_parameter(ed_patient_ids))
triage_raw = query_dataframe(f"SELECT * FROM `{ED}.triage`")
assert edstays_raw.stay_id.is_unique and patients_raw.subject_id.is_unique
assert admissions_raw.hadm_id.is_unique and triage_raw.stay_id.is_unique
flow = [{"step": "All ED encounters", "encounters": len(edstays_raw), "patients": edstays_raw.subject_id.nunique()}]

In [ ]:
# Create a working copy of triage
triage = triage_raw.copy()

# Convert vital signs to numbers
vitals = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]

for column in vitals:
    triage[column] = pd.to_numeric(triage[column], errors="coerce")

# Check whether conversion created any extra missing values
print("Extra missing values after conversion:")
print(triage[vitals].isna().sum() - triage_raw[vitals].isna().sum())

# Show numeric summaries
display(triage[vitals].describe())

In [ ]:
# Keep ED encounters linked to a hospital admission
cohort = edstays_raw[edstays_raw["hadm_id"].notna()].copy()

print("Admitted ED encounters:", len(cohort))

# Select the demographic fields we need
demographics = patients_raw[
    ["subject_id", "anchor_age", "anchor_year", "anchor_year_group"]
]

# Add demographics to each encounter
cohort = cohort.merge(
    demographics,
    on="subject_id",
    how="left",
    validate="many_to_one"
)

# Calculate age at ED arrival
cohort["intime"] = pd.to_datetime(cohort["intime"])

cohort["age"] = (
    cohort["anchor_age"] + cohort["intime"].dt.year - cohort["anchor_year"]
)

# Inspect age before applying the adult restriction
print("Encounters after merge:", len(cohort))
print("Unique ED stays:", cohort["stay_id"].nunique())
print("Encounters aged under 18:", (cohort["age"] < 18).sum())

display(cohort["age"].describe())
flow.append({"step": 'Hospital-admitted ED encounters', "encounters": len(cohort), "patients": cohort.subject_id.nunique()})

In [ ]:
# Keep only adults (It wont remove anyone)
cohort = cohort[cohort["age"] >= 18].copy()

# Select hospital admission and discharge times
hospital_times = admissions_raw[
    ["subject_id", "hadm_id", "admittime", "dischtime"]
]

# Add hospital times
cohort = cohort.merge(
    hospital_times,
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one"
)

# Convert time columns to datetime
time_columns = ["intime", "outtime", "admittime", "dischtime"]

for column in time_columns:
    cohort[column] = pd.to_datetime(cohort[column])

# Check missing times
print("Missing times:")
print(cohort[time_columns].isna().sum())

# Check timing problems
bad_ed_times = cohort["outtime"] < cohort["intime"]
bad_discharge_times = cohort["dischtime"] <= cohort["intime"]

print("\nED departure before arrival:", bad_ed_times.sum())
print("Hospital discharge at or before ED arrival:", bad_discharge_times.sum())

print("\nEncounters after merge:", len(cohort))
print("Unique ED stays:", cohort["stay_id"].nunique())
flow.append({"step": 'Adults', "encounters": len(cohort), "patients": cohort.subject_id.nunique()})

In [ ]:
# Keep only encounters with valid timing
valid_times = (
    (cohort["outtime"] >= cohort["intime"])
    & (cohort["dischtime"] > cohort["intime"])
)

print("Excluded for invalid timing:", (~valid_times).sum())

cohort = cohort[valid_times].copy()

# Add the triage information converted to numeric earlier
cohort = cohort.merge(
    triage,
    on=["subject_id", "stay_id"],
    how="left",
    validate="one_to_one"
)

# Check the cohort size
print("Remaining encounters:", len(cohort))
print("Unique ED stays:", cohort["stay_id"].nunique())
print("Unique patients:", cohort["subject_id"].nunique())
flow.append({"step": 'Valid encounter times', "encounters": len(cohort), "patients": cohort.subject_id.nunique()})

In [ ]:
diagnoses_raw = query_dataframe(f"""SELECT subject_id, hadm_id, seq_num, icd_code, icd_version
FROM `{HOSP}.diagnoses_icd` WHERE subject_id IN UNNEST(@patient_ids)""", patient_parameter(cohort.subject_id.unique()))

In [ ]:
# Create a working copy
diagnoses = diagnoses_raw.copy()

# Standardise the code text
diagnoses["icd_code"] = diagnoses["icd_code"].str.strip().str.upper()

# Flag ESKD codes
diagnoses["eskd"] = (
    ((diagnoses["icd_version"] == 9) & (diagnoses["icd_code"] == "5856"))
    | ((diagnoses["icd_version"] == 10) & (diagnoses["icd_code"] == "N186"))
)

# Flag chronic dialysis status codes
diagnoses["dialysis"] = (
    ((diagnoses["icd_version"] == 9)
     & diagnoses["icd_code"].isin(["V451", "V4511"]))
    | ((diagnoses["icd_version"] == 10)
       & (diagnoses["icd_code"] == "Z992"))
)

# Keep records with either flag
renal_diagnoses = diagnoses[
    diagnoses["eskd"] | diagnoses["dialysis"]
].copy()

# Summarise the matching codes
print("Matching diagnosis records:", len(renal_diagnoses))
print("Patients with matching codes:", renal_diagnoses["subject_id"].nunique())

print("\nRecords per code:")
print(
    renal_diagnoses.groupby(["icd_version", "icd_code"]).size()
)

# Add discharge times to the matching diagnosis records
renal_history = renal_diagnoses.merge(
    admissions_raw[["subject_id", "hadm_id", "dischtime"]],
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one"
)

# Rename columns to distinguish historical and current admissions
renal_history = renal_history.rename(columns={
    "hadm_id": "history_hadm_id",
    "dischtime": "history_dischtime"
})

renal_history["history_dischtime"] = pd.to_datetime(
    renal_history["history_dischtime"]
)

print("Missing historical discharge times:",
      renal_history["history_dischtime"].isna().sum())

# Match each ED encounter to the patient's renal diagnosis history
history_check = cohort[
    ["subject_id", "stay_id", "hadm_id", "intime"]
].merge(
    renal_history,
    on="subject_id",
    how="inner"
)

# Keep evidence from a different admission completed before ED arrival
prior_history = history_check[
    (history_check["history_hadm_id"] != history_check["hadm_id"])
    & (history_check["history_dischtime"] < history_check["intime"])
].copy()

# Get the ED stays with each type of prior evidence
eskd_stays = prior_history.loc[
    prior_history["eskd"], "stay_id"
].unique()

dialysis_stays = prior_history.loc[
    prior_history["dialysis"], "stay_id"
].unique()

# Add encounter-level flags
cohort["prior_eskd"] = cohort["stay_id"].isin(eskd_stays)
cohort["prior_dialysis"] = cohort["stay_id"].isin(dialysis_stays)

# Count encounters, not diagnosis records
print("Prior ESKD:", cohort["prior_eskd"].sum())
print("Prior dialysis:", cohort["prior_dialysis"].sum())

print("Both:",
      (cohort["prior_eskd"] & cohort["prior_dialysis"]).sum())

print("Either:",
      (cohort["prior_eskd"] | cohort["prior_dialysis"]).sum())

# Flag encounters to exclude
renal_exclusion = cohort["prior_eskd"] | cohort["prior_dialysis"]

# Keep excluded encounters separately for our records
renal_excluded = cohort[renal_exclusion].copy()

# Keep encounters without either exclusion
cohort = cohort[~renal_exclusion].copy()

# Check the remaining cohort
print("Excluded encounters:", len(renal_excluded))
print("Remaining encounters:", len(cohort))
print("Unique ED stays:", cohort["stay_id"].nunique())
print("Unique patients:", cohort["subject_id"].nunique())
flow.append({"step": "After prior ESKD/dialysis exclusions", "encounters": len(cohort), "patients": cohort.subject_id.nunique()})

In [ ]:
lab_items = query_dataframe(f"SELECT * FROM `{HOSP}.d_labitems`")
display(lab_items.loc[lab_items.itemid.isin([50912, 52546, 52024]), ["itemid", "label", "fluid", "category"]])
creatinine_raw = query_dataframe(f"""SELECT labevent_id, subject_id, hadm_id, specimen_id,
itemid, charttime, storetime, value, valuenum, valueuom
FROM `{HOSP}.labevents` WHERE subject_id IN UNNEST(@patient_ids)
AND itemid IN (50912, 52546, 52024)""", patient_parameter(cohort.subject_id.unique()))
cohort, sample_results = derive_phenotype(cohort, creatinine_raw)
assert cohort.stay_id.is_unique
labelled = cohort.loc[cohort.aki_label.notna()].copy()
flow.append({"step": "Labelled modelling population", "encounters": len(labelled), "patients": labelled.subject_id.nunique()})
save_csv(pd.DataFrame(flow), "cohort_flow.csv")
save_csv(cohort.label_state.value_counts().rename_axis("label_state").reset_index(name="encounters"), "outcome_counts.csv")
display(pd.DataFrame(flow))

In [ ]:
model_data = labelled.copy()
vitals = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]
# Preserve the original values so this block can be rerun
columns = vitals + ["acuity"]

for column in columns:
    model_data[column + "_raw"] = labelled[column]

# Clean temperature
temp = model_data["temperature_raw"]

model_data["temperature"] = temp.where((temp > 70) & (temp < 120))

# Convert assumed Celsius values to Fahrenheit
celsius = temp.between(30, 43)

model_data.loc[celsius, "temperature"] = (
    temp[celsius] * 9 / 5 + 32
)

# Clean the other vital signs using the original limits
limits = {
    "heartrate": 300,
    "resprate": 70,
    "sbp": 400,
    "dbp": 300
}

for column, upper_limit in limits.items():
    values = model_data[column + "_raw"]
    model_data[column] = values.where(
        (values > 0) & (values < upper_limit)
    )

# Oxygen saturation allows 100
oxygen = model_data["o2sat_raw"]
model_data["o2sat"] = oxygen.where((oxygen > 0) & (oxygen <= 100))

# Keep acuity values from 1 to 5
acuity = pd.to_numeric(model_data["acuity_raw"], errors="coerce")
model_data["acuity"] = acuity.where(acuity.between(1, 5))

# Check the changes
print("Temperature values converted:", celsius.sum())

print("\nAdditional missing values after cleaning:")
print(
    model_data[columns].isna().sum()
    - labelled[columns].isna().sum()
)

display(model_data[vitals].describe())

# Preserve the original text
model_data["pain_raw"] = labelled["pain"]
model_data["chiefcomplaint_raw"] = labelled["chiefcomplaint"]

# Remove spaces around pain entries
pain = model_data["pain_raw"].astype("string").str.strip()

# Accept 0–10, including entries such as "5.0"
numeric_pain = pain.str.fullmatch(r"([0-9]|10)(\.0+)?", na=False)

model_data["pain_numeric"] = pd.to_numeric(
    pain.where(numeric_pain), errors="coerce"
)

# Distinguish missing pain from other recorded entries
missing_pain = pain.isna() | pain.eq("").fillna(False)

model_data["pain_status"] = "other_recorded"
model_data.loc[missing_pain, "pain_status"] = "missing"
model_data.loc[numeric_pain, "pain_status"] = "numeric"

# Trim chief complaints and treat blank text as missing
complaint = model_data["chiefcomplaint_raw"].astype("string").str.strip()
model_data["chiefcomplaint"] = complaint.replace("", pd.NA)

# Check the results
print("Pain status:")
print(model_data["pain_status"].value_counts())

print("\nNumeric pain summary:")
print(model_data["pain_numeric"].describe())

print("\nMissing chief complaints:",
      model_data["chiefcomplaint"].isna().sum())

print("\nMost common other pain entries:")
print(pain[model_data["pain_status"] == "other_recorded"].value_counts().head(15))
modelling = model_data[["subject_id", "stay_id", "hadm_id"] + PREDICTORS + ["aki_label"]].copy()
modelling["aki_label"] = modelling["aki_label"].astype(int)
modelling["aki_stage2_label"] = model_data["observed_stage2_positive"].astype(int)
# Normalize pandas extension dtypes for sklearn; preserve literal text such as "N/A".
for column in NUMERIC:
    modelling[column] = pd.to_numeric(modelling[column], errors="coerce").astype(float)
for column in CATEGORICAL + [TEXT]:
    modelling[column] = modelling[column].astype(object).where(modelling[column].notna(), np.nan)
audit = cohort.copy()
raw_labelled = labelled.copy()
observed_counts = {"eligible_encounters": len(cohort), "modelling_encounters": len(modelling),
    "modelling_patients": modelling.subject_id.nunique(), "primary_positive": int(modelling.aki_label.sum()),
    "secondary_positive": int(modelling.aki_stage2_label.sum())}
expected_counts = {"eligible_encounters": 195340, "modelling_encounters": 113097,
    "modelling_patients": 66090, "primary_positive": 7866, "secondary_positive": 2254}
write_json(CHECKS / "historical_count_comparison.json", {"observed": observed_counts,
    "historical_reference": expected_counts, "matches": observed_counts == expected_counts})
if REQUIRE_HISTORICAL_COUNTS:
    assert observed_counts == expected_counts, "Fresh source cohort differs from historical counts; inspect source versions and query audit before proceeding."
assert (modelling.aki_stage2_label <= modelling.aki_label).all()
print(observed_counts)

## Part 2 — phenotype review, descriptive statistics, patient split and preprocessing (Person 2)

P1's production phenotype is checked against P2's independent boundary-case implementation. The fresh
eligible cohort also undergoes the original row-level timing, label and severity checks. Literal text is
preserved directly from BigQuery; no CSV parsing can silently convert a recorded `N/A` pain entry to missing.
The historical cohort counts are a drift check, not fabricated output. A mismatch stops the default run.

In [ ]:
a = audit.copy()
hours = lambda td: td.dt.total_seconds() / 3600
tol = 1e-6
has_sample = a["current_sample_count"] > 0
pos = a["label_state"] == "positive"
neg = a["label_state"] == "negative"
ind = a["label_state"] == "indeterminate"
nouse = a["label_state"] == "no_usable_current_creatinine"
expected_endpoint = np.minimum(a["intime"] + pd.Timedelta(hours=24), a["dischtime"])

rules = {
    # Cohort eligibility
    "Age under 18": (a["age"] < 18),
    "Hospital discharge at or before ED arrival": (a["dischtime"] <= a["intime"]),
    "ED departure before arrival": (a["outtime"] < a["intime"]),
    "Prior ESKD or dialysis retained": (a["prior_eskd"] | a["prior_dialysis"]),
    # Observation window
    "endpoint_time != min(arrival + 24 h, discharge)": (a["endpoint_time"] != expected_endpoint),
    "followup_hours inconsistent with endpoint": ((hours(a["endpoint_time"] - a["intime"]) - a["followup_hours"]).abs() > tol),
    "followup_hours outside (0, 24]": ((a["followup_hours"] <= 0) | (a["followup_hours"] > 24)),
    # Sampling fields
    "Sample count > 0 but no first sample time": (has_sample & a["first_sample_time"].isna()),
    "Sample count = 0 but a first sample time": (~has_sample & a["first_sample_time"].notna()),
    "First sample before ED arrival": (a["first_sample_time"] < a["intime"]),
    "Last sample after endpoint": (a["last_sample_time"] > a["endpoint_time"]),
    "One sample but first != last time": ((a["current_sample_count"] == 1) & (a["first_sample_time"] != a["last_sample_time"])),
    "first_sample_delay_hours inconsistent": ((hours(a["first_sample_time"] - a["intime"]) - a["first_sample_delay_hours"]).abs() > tol),
    "sampling_span_hours inconsistent": ((hours(a["last_sample_time"] - a["first_sample_time"]) - a["sampling_span_hours"]).abs() > tol),
    "endpoint_gap_hours inconsistent": ((hours(a["endpoint_time"] - a["last_sample_time"]) - a["endpoint_gap_hours"]).abs() > tol),
    # Primary outcome
    "observed_positive disagrees with label_state": (a["observed_positive"] != pos),
    "Positive without a current sample": (pos & ~has_sample),
    "Positive time missing": (pos & a["first_positive_time"].isna()),
    "Positive time outside [arrival, endpoint]": (pos & ((a["first_positive_time"] < a["intime"]) | (a["first_positive_time"] > a["endpoint_time"]))),
    "Positive time outside [first, last] sample": (pos & ((a["first_positive_time"] < a["first_sample_time"]) | (a["first_positive_time"] > a["last_sample_time"]))),
    "hours_to_first_positive inconsistent": (pos & ((hours(a["first_positive_time"] - a["intime"]) - a["hours_to_first_positive"]).abs() > tol)),
    "First sample 'positive' but positive time != first sample time": ((a["first_sample_status"] == "positive") & (a["first_positive_time"] != a["first_sample_time"])),
    "Non-positive with a positive time": (~pos & a["first_positive_time"].notna()),
    # Operational negatives and unresolved states
    "Negative with < 2 current samples": (neg & (a["current_sample_count"] < 2)),
    "Negative without a timed comparison": (neg & ~a["has_timed_comparison"]),
    "Negative with an observed rise": (neg & a["observed_positive"]),
    "serial_no_rise disagrees with negative state": (a["serial_no_rise"] != neg),
    "Indeterminate with 0 samples": (ind & ~has_sample),
    "Indeterminate meeting the negative rule": (ind & (a["current_sample_count"] >= 2) & a["has_timed_comparison"]),
    "No-usable-creatinine state with samples": (nouse & has_sample),
    "first_sample_status 'no current sample' disagrees with count": ((a["first_sample_status"] == "no current sample") != ~has_sample),
    # Binary label
    "aki_label != 1 for positive": (pos & (a["aki_label"] != 1)),
    "aki_label != 0 for negative": (neg & (a["aki_label"] != 0)),
    "Unresolved encounter with a binary label": ((ind | nouse) & a["aki_label"].notna()),
    # Severity
    "Stage missing for positive": (pos & a["highest_creatinine_stage"].isna()),
    "Stage present for non-positive": (~pos & a["highest_creatinine_stage"].notna()),
    "Stage outside 1-3": (a["highest_creatinine_stage"].notna() & ~a["highest_creatinine_stage"].isin([1, 2, 3])),
    "Stage >= 2 flag disagrees with highest stage": (a["observed_stage2_positive"] != (a["highest_creatinine_stage"] >= 2).fillna(False)),
    "Stage >= 2 outside primary positives": (a["observed_stage2_positive"] & ~pos),
    "Stage >= 2 time before first positive time": (a["first_stage2_time"] < a["first_positive_time"]),
    "Stage >= 2 time after endpoint": (a["first_stage2_time"] > a["endpoint_time"]),
}

rule_checks = pd.DataFrame(
    [{"rule": k, "violating_encounters": int(v.fillna(False).sum())} for k, v in rules.items()]
)
rule_checks.to_csv(OUT_DIR / "qa_rule_checks.csv", index=False, encoding="utf-8-sig")
print("Encounters checked:", len(a))
print("All rule checks pass:", (rule_checks["violating_encounters"] == 0).all())
rule_checks
assert rule_checks["violating_encounters"].eq(0).all()

In [ ]:
recleaned = clean_predictors(raw_labelled)

cleaning_check = []
for col in predictor_columns:
    mine, theirs = recleaned[col], modelling[col]
    if pd.api.types.is_numeric_dtype(theirs):
        mine = mine.astype(float)
        same = (np.isclose(mine, theirs, rtol=0, atol=1e-9) | (mine.isna() & theirs.isna()))
    else:
        same = (mine.astype("string").fillna("<NA>") == theirs.astype("string").fillna("<NA>"))
    cleaning_check.append({"predictor": col, "rows_disagreeing": int((~same).sum())})
cleaning_check = pd.DataFrame(cleaning_check)
cleaning_check.to_csv(OUT_DIR / "qa_cleaning_reproduction.csv", index=False, encoding="utf-8-sig")


assert cleaning_check["rows_disagreeing"].eq(0).all()

In [ ]:
def fmt_n(n, d):
    return f"{n:,} ({100 * n / d:.1f}%)" if d else "–"

def med_iqr(x):
    x = x.dropna()
    if len(x) == 0:
        return "–"
    q1, q2, q3 = x.quantile([0.25, 0.5, 0.75])
    return f"{q2:.1f} [{q1:.1f}–{q3:.1f}]"

def smd_cont(x1, x0):
    x1, x0 = x1.dropna().astype(float), x0.dropna().astype(float)
    sd = np.sqrt((x1.var() + x0.var()) / 2)
    return (x1.mean() - x0.mean()) / sd if sd > 0 else np.nan

def smd_bin(b1, b0):
    p1, p0 = b1.mean(), b0.mean()
    sd = np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2)
    return (p1 - p0) / sd if sd > 0 else np.nan

def describe_groups(df, groups, continuous, categorical, smd_pairs, labels=None):
    """groups: {column label: boolean mask}. smd_pairs: {smd label: (group a, reference group b)}."""
    labels = labels or {}
    rows = []
    header = {g: df[m] for g, m in groups.items()}
    rows.append({"variable": "Encounters, n", "level": "",
                 **{g: f"{len(d):,}" for g, d in header.items()},
                 **{k: "" for k in smd_pairs}})
    rows.append({"variable": "Patients, n (not additive across columns)", "level": "",
                 **{g: f"{d['subject_id'].nunique():,}" for g, d in header.items()},
                 **{k: "" for k in smd_pairs}})
    for var in continuous:
        name = labels.get(var, var)
        rows.append({"variable": name, "level": "median [IQR]",
                     **{g: med_iqr(d[var]) for g, d in header.items()},
                     **{k: f"{smd_cont(header[a][var], header[b][var]):.3f}" for k, (a, b) in smd_pairs.items()}})
        rows.append({"variable": name, "level": "missing",
                     **{g: fmt_n(int(d[var].isna().sum()), len(d)) for g, d in header.items()},
                     **{k: f"{smd_bin(header[a][var].isna(), header[b][var].isna()):.3f}" for k, (a, b) in smd_pairs.items()}})
    for var in categorical:
        name = labels.get(var, var)
        levels = df[var].astype("string").fillna("Missing").value_counts().index.tolist()
        for lev in levels:
            ind = {g: d[var].astype("string").fillna("Missing") == lev for g, d in header.items()}
            rows.append({"variable": name, "level": lev,
                         **{g: fmt_n(int(ind[g].sum()), len(header[g])) for g in header},
                         **{k: f"{smd_bin(ind[a], ind[b]):.3f}" for k, (a, b) in smd_pairs.items()}})
    return pd.DataFrame(rows)

labels = {
    "age": "Age, years", "gender": "Gender", "arrival_transport": "Arrival transport",
    "temperature": "Temperature, °F", "heartrate": "Heart rate, /min", "resprate": "Respiratory rate, /min",
    "o2sat": "Oxygen saturation, %", "sbp": "Systolic BP, mmHg", "dbp": "Diastolic BP, mmHg",
    "acuity_cat": "Triage acuity (ESI)", "acuity": "Triage acuity (ESI)", "pain_numeric": "Pain score (0–10)",
    "pain_status": "Pain entry", "all_vitals_missing": "All six vital signs missing",
    "race_group": "Race (descriptive only, not a predictor)", "anchor_year_group": "Anchor year group",
    "followup_hours": "Follow-up window, h", "ed_los_hours": "ED length of stay, h",
    "hosp_los_days": "Hospital length of stay, days", "discharged_lt24h": "Hospital discharge < 24 h after arrival",
}

md = modelling.copy()
md["acuity_cat"] = md["acuity"].map(lambda v: f"{int(v)}" if pd.notna(v) else pd.NA)
md["all_vitals_missing"] = np.where(md[vitals].isna().all(axis=1), "Yes", "No")

groups = {
    f"Operational negatives (N={int((md['aki_label'] == 0).sum()):,})": md["aki_label"] == 0,
    f"Primary positives (N={int((md['aki_label'] == 1).sum()):,})": md["aki_label"] == 1,
    f"  of which stage ≥2 (N={int((md['aki_stage2_label'] == 1).sum()):,})": md["aki_stage2_label"] == 1,
    f"All labelled (N={len(md):,})": pd.Series(True, index=md.index),
}
g = list(groups)
table1 = describe_groups(
    md, groups,
    continuous=["age"] + vitals + ["pain_numeric"],
    categorical=["gender", "arrival_transport", "acuity_cat", "pain_status", "all_vitals_missing"],
    smd_pairs={"SMD positive vs negative": (g[1], g[0])},
    labels=labels,
)
table1.to_csv(OUT_DIR / "table1_characteristics_pos_vs_neg.csv", index=False, encoding="utf-8-sig")
table1

In [ ]:
miss_rows = []
neg_m, pos_m, s2_m = md["aki_label"] == 0, md["aki_label"] == 1, md["aki_stage2_label"] == 1
for col in predictor_columns + ["all six vital signs"]:
    if col == "all six vital signs":
        miss = md[vitals].isna().all(axis=1)
    else:
        miss = md[col].isna()
    miss_rows.append({
        "predictor": col,
        f"Negatives missing (N={neg_m.sum():,})": fmt_n(int(miss[neg_m].sum()), int(neg_m.sum())),
        f"Positives missing (N={pos_m.sum():,})": fmt_n(int(miss[pos_m].sum()), int(pos_m.sum())),
        f"Stage ≥2 missing (N={s2_m.sum():,})": fmt_n(int(miss[s2_m].sum()), int(s2_m.sum())),
        f"All labelled missing (N={len(md):,})": fmt_n(int(miss.sum()), len(md)),
        "SMD positive vs negative": round(smd_bin(miss[pos_m], miss[neg_m]), 3),
        "AKI rate if missing": f"{md.loc[miss, 'aki_label'].mean():.1%}" if miss.any() else "–",
        "AKI rate if observed": f"{md.loc[~miss, 'aki_label'].mean():.1%}",
    })
missingness = pd.DataFrame(miss_rows)
missingness.to_csv(OUT_DIR / "table2_missingness_by_outcome.csv", index=False, encoding="utf-8-sig")
print("Note: pain_numeric is missing for both 'missing' and 'other_recorded' pain entries; "
      "pain_status keeps them apart.")
missingness

In [ ]:
audit_raw_full = audit.copy()
cleaned_full = pd.concat([audit_raw_full[["stay_id"]], clean_predictors(audit_raw_full)], axis=1)

# The cleaned audit predictors must equal modelling_data for labelled encounters
chk = modelling[["stay_id"] + predictor_columns].merge(cleaned_full, on="stay_id", suffixes=("", "_re"))
num_cols = ["age", "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "acuity", "pain_numeric"]
bad = sum(int((~(np.isclose(chk[c], chk[c + "_re"].astype(float), atol=1e-9)
                 | (chk[c].isna() & chk[c + "_re"].isna()))).sum()) for c in num_cols)
assert bad == 0
print("Cleaned audit predictors disagreeing with modelling_data (labelled rows):", bad)

def race_group(r):
    r = str(r).upper()
    if r.startswith("WHITE") or r == "PORTUGUESE": return "White"
    if r.startswith("BLACK"): return "Black"
    if r.startswith("HISPANIC") or r.startswith("SOUTH AMERICAN"): return "Hispanic/Latino"
    if r.startswith("ASIAN"): return "Asian"
    if r in ("UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER"): return "Unknown/declined"
    return "Other"

full = audit[["stay_id", "subject_id", "label_state", "race", "anchor_year_group", "followup_hours",
              "intime", "outtime", "admittime", "dischtime"]].merge(cleaned_full, on="stay_id", validate="one_to_one")
full["race_group"] = full["race"].map(race_group)
full["acuity_cat"] = full["acuity"].map(lambda v: f"{int(v)}" if pd.notna(v) else pd.NA)
full["all_vitals_missing"] = np.where(full[vitals].isna().all(axis=1), "Yes", "No")
full["discharged_lt24h"] = np.where(full["followup_hours"] < 24, "Yes", "No")
full["ed_los_hours"] = (full["outtime"] - full["intime"]).dt.total_seconds() / 3600
full["hosp_los_days"] = (full["dischtime"] - full["admittime"]).dt.total_seconds() / 86400

is_lab = full["label_state"].isin(["positive", "negative"])
is_ind = full["label_state"] == "indeterminate"
is_nou = full["label_state"] == "no_usable_current_creatinine"
sel_groups = {
    f"Labelled (N={is_lab.sum():,})": is_lab,
    f"Indeterminate (N={is_ind.sum():,})": is_ind,
    f"No usable creatinine (N={is_nou.sum():,})": is_nou,
    f"All eligible (N={len(full):,})": pd.Series(True, index=full.index),
}
sg = list(sel_groups)
table3 = describe_groups(
    full, sel_groups,
    continuous=["age"] + vitals + ["pain_numeric", "followup_hours", "ed_los_hours", "hosp_los_days"],
    categorical=["gender", "arrival_transport", "acuity_cat", "pain_status", "all_vitals_missing",
                 "discharged_lt24h", "race_group", "anchor_year_group"],
    smd_pairs={"SMD indeterminate vs labelled": (sg[1], sg[0]),
               "SMD no-creatinine vs labelled": (sg[2], sg[0])},
    labels=labels,
)
table3.to_csv(OUT_DIR / "table3_labelled_vs_excluded.csv", index=False, encoding="utf-8-sig")
table3

In [ ]:
def dist_row(name, series_by_group, fn):
    return {"measure": name, **{gname: fn(s) for gname, s in series_by_group.items()}}

state_order = ["positive", "negative", "indeterminate", "no_usable_current_creatinine"]
state_names = {"positive": "Primary positive", "negative": "Operational negative",
               "indeterminate": "Indeterminate", "no_usable_current_creatinine": "No usable creatinine"}
by_state = {f"{state_names[s]} (N={(audit['label_state'] == s).sum():,})": audit[audit["label_state"] == s]
            for s in state_order}

asc = []
asc.append(dist_row("Current samples, median [IQR]", {k: d["current_sample_count"] for k, d in by_state.items()}, med_iqr))
for lev, f in [("0", lambda x: x == 0), ("1", lambda x: x == 1), ("2", lambda x: x == 2),
               ("3", lambda x: x == 3), ("≥4", lambda x: x >= 4)]:
    asc.append(dist_row(f"Current samples = {lev}",
                        {k: d["current_sample_count"] for k, d in by_state.items()},
                        lambda s, f=f: fmt_n(int(f(s).sum()), len(s))))
asc.append(dist_row("First-sample delay from arrival, h", {k: d["first_sample_delay_hours"] for k, d in by_state.items()}, med_iqr))
asc.append(dist_row("Sampling span (first to last sample), h", {k: d["sampling_span_hours"] for k, d in by_state.items()}, med_iqr))
for lab, lo, hi in [("Sampling span < 6 h", 0, 6), ("Sampling span 6–<12 h", 6, 12), ("Sampling span ≥ 12 h", 12, np.inf)]:
    asc.append(dist_row(f"{lab} (among ≥2 samples)",
                        {k: d.loc[d["current_sample_count"] >= 2, "sampling_span_hours"] for k, d in by_state.items()},
                        lambda s, lo=lo, hi=hi: fmt_n(int(((s >= lo) & (s < hi)).sum()), len(s)) if len(s) else "–"))
asc.append(dist_row("Gap from last sample to endpoint, h", {k: d["endpoint_gap_hours"] for k, d in by_state.items()}, med_iqr))
asc.append(dist_row("Follow-up window, h", {k: d["followup_hours"] for k, d in by_state.items()}, med_iqr))
asc.append(dist_row("Full 24 h window", {k: d["followup_hours"] for k, d in by_state.items()},
                    lambda s: fmt_n(int((s == 24).sum()), len(s))))
for status in ["positive", "no qualifying rise", "unassessable", "no current sample"]:
    asc.append(dist_row(f"First sample status: {status}", {k: d["first_sample_status"] for k, d in by_state.items()},
                        lambda s, st=status: fmt_n(int((s == st).sum()), len(s))))
asc.append(dist_row("Hours from arrival to first qualifying sample", {k: d["hours_to_first_positive"] for k, d in by_state.items()}, med_iqr))
ascertainment = pd.DataFrame(asc)
ascertainment.to_csv(OUT_DIR / "table4_ascertainment_by_state.csv", index=False, encoding="utf-8-sig")
ascertainment

In [ ]:
p_ = audit[audit["label_state"] == "positive"].copy()
grp_names = {"positive": "Already positive at first sample",
             "no qualifying rise": "Assessable, no rise at first sample, later positive",
             "unassessable": "Unassessable at first sample, later positive"}
fs_rows = []
for key, name in grp_names.items():
    d = p_[p_["first_sample_status"] == key]
    fs_rows.append({
        "group": name,
        "encounters": len(d),
        "% of positives": f"{len(d) / len(p_):.1%}",
        "patients (not additive)": d["subject_id"].nunique(),
        "hours to first qualifying sample, median [IQR]": med_iqr(d["hours_to_first_positive"]),
        "first-sample delay h, median [IQR]": med_iqr(d["first_sample_delay_hours"]),
        "current samples, median [IQR]": med_iqr(d["current_sample_count"]),
        "single current sample": fmt_n(int((d["current_sample_count"] == 1).sum()), len(d)),
        "stage 1": fmt_n(int((d["highest_creatinine_stage"] == 1).sum()), len(d)),
        "stage 2": fmt_n(int((d["highest_creatinine_stage"] == 2).sum()), len(d)),
        "stage 3": fmt_n(int((d["highest_creatinine_stage"] == 3).sum()), len(d)),
        "stage ≥2": fmt_n(int(d["observed_stage2_positive"].sum()), len(d)),
    })
first_sample_groups = pd.DataFrame(fs_rows)
first_sample_groups.to_csv(OUT_DIR / "table5_first_sample_groups_positives.csv", index=False, encoding="utf-8-sig")
first_sample_groups.T

In [ ]:
flags = modelling[["subject_id", "stay_id", "aki_label", "aki_stage2_label"]].merge(
    audit[["stay_id", "current_sample_count", "first_sample_status", "sampling_span_hours",
           "followup_hours"]], on="stay_id", validate="one_to_one")
flags["S1_has_any_vital"] = ~modelling[vitals].isna().all(axis=1).values
flags["S2_complete_case"] = modelling[predictor_columns].notna().all(axis=1).values
flags["S3_excl_positive_at_first_sample"] = ~((flags["aki_label"] == 1) & (flags["first_sample_status"] == "positive"))
flags["S4_two_or_more_samples"] = flags["current_sample_count"] >= 2
flags["S5_dense_negatives"] = (flags["aki_label"] == 1) | (flags["sampling_span_hours"] >= 12)
flags["S6_full_24h_followup"] = flags["followup_hours"] == 24

subset_cols = [c for c in flags.columns if c.startswith("S")]
sub_rows = [{"subset": "All labelled", "encounters": len(flags), "patients": flags["subject_id"].nunique(),
             "primary positives": int(flags["aki_label"].sum()),
             "primary prevalence": f"{flags['aki_label'].mean():.2%}",
             "stage ≥2 positives": int(flags["aki_stage2_label"].sum()),
             "stage ≥2 prevalence": f"{flags['aki_stage2_label'].mean():.2%}"}]
for c in subset_cols:
    d = flags[flags[c]]
    sub_rows.append({"subset": c, "encounters": len(d), "patients": d["subject_id"].nunique(),
                     "primary positives": int(d["aki_label"].sum()),
                     "primary prevalence": f"{d['aki_label'].mean():.2%}",
                     "stage ≥2 positives": int(d["aki_stage2_label"].sum()),
                     "stage ≥2 prevalence": f"{d['aki_stage2_label'].mean():.2%}"})
subset_counts = pd.DataFrame(sub_rows)
subset_counts.to_csv(OUT_DIR / "sensitivity_subset_counts.csv", index=False, encoding="utf-8-sig")

# Row-level flags: identifiers + booleans only (no predictors, no outcome times)
flags_out = flags[["subject_id", "stay_id"] + subset_cols]
flags_out.to_csv(OUT_DIR / "sensitivity_subset_flags.csv", index=False, encoding="utf-8-sig")
print("Row-level flags saved:", flags_out.shape, "- private storage only; NOT model inputs.")
subset_counts

In [ ]:
df = modelling.copy()
patients = df.groupby("subject_id").agg(encounters=("stay_id", "size"),
    any_aki=("aki_label", "max"), any_stage2=("aki_stage2_label", "max")).reset_index()
patients["stratum"] = np.select([patients.any_stage2.eq(1), patients.any_aki.eq(1)],
    ["stage2", "stage1_only"], "none")
train_p, hold_p = train_test_split(patients, test_size=0.30, stratify=patients.stratum, random_state=SPLIT_SEED)
val_p, test_p = train_test_split(hold_p, test_size=0.50, stratify=hold_p.stratum, random_state=SPLIT_SEED)
partition_of = pd.concat([pd.Series(name, index=group.subject_id) for name, group in
    [("train", train_p), ("validation", val_p), ("test", test_p)]])
df["partition"] = df.subject_id.map(partition_of)
assert df.partition.notna().all() and df.stay_id.is_unique
assert df.groupby("subject_id").partition.nunique().eq(1).all()
data = {part: df.loc[df.partition.eq(part)].sort_values("stay_id").reset_index(drop=True)
        for part in ("train", "validation", "test")}
split_summary = pd.DataFrame([{"partition": part, "encounters": len(d), "patients": d.subject_id.nunique(),
    "primary_positive": int(d.aki_label.sum()), "secondary_positive": int(d.aki_stage2_label.sum())}
    for part, d in data.items()])
save_csv(split_summary, "split_summary.csv")
save_csv(df[["subject_id", "stay_id", "partition"]], "split_manifest.csv", PRIVATE)
display(split_summary)
X = {part: d[PREDICTORS].copy() for part, d in data.items()}
assert set(PREDICTORS).isdisjoint({"subject_id", "stay_id", "hadm_id", "aki_label", "aki_stage2_label"})
fitted, feature_names, matrices = {}, {}, {}
with threadpool_limits(limits=1):
    for name, builder in PREPROCESSORS.items():
        pre = builder().fit(X["train"])
        fitted[name] = pre
        feature_names[name] = list(pre.get_feature_names_out())
        # Test transformation is deferred until model/threshold freezing.
        matrices[name] = {part: pre.transform(X[part]) for part in ("train", "validation")}
        print(name, {part: matrix.shape for part, matrix in matrices[name].items()})
imputer = fitted["lr_structured"].named_transformers_["num"].named_steps["impute"]
assert np.allclose(imputer.statistics_, X["train"][NUMERIC].median().to_numpy())
assert np.allclose(matrices["lr_structured"]["train"][:, :len(NUMERIC)].mean(axis=0), 0, atol=1e-8)
for name in ("lr_structured", "lr_text"):
    for matrix in matrices[name].values():
        assert np.isfinite(matrix.data if sp.issparse(matrix) else matrix).all()
assert fitted["lr_text"].named_transformers_["text"].named_steps["tfidf"].vocabulary_ == fitted["tree_text"].named_transformers_["text"].named_steps["tfidf"].vocabulary_
write_json(CHECKS / "preprocessing.json", {"fit_partition": "train", "split_seed": SPLIT_SEED,
    "features": feature_names, "test_transformed": False, "imputation_medians": imputer.statistics_.tolist()})

## Part 3 — four baselines per outcome (Person 3)

The dummy predicts the training prevalence. Age/gender LR is unpenalized. Structured and text-inclusive
LR select `C ∈ {0.001, 0.01, 0.1, 1, 10, 100}` by validation AP (ties within 1e-12 favour smaller C).
The text model separates the gain from adding text from the gain from changing the algorithm.
All selected models remain fitted on training patients only; convergence warnings stop the run.

In [ ]:
TARGETS = ["aki_label", "aki_stage2_label"]
MODEL_NAMES = ["dummy", "age_gender_lr", "structured_lr", "structured_text_lr",
    "age_gender_xgboost", "structured_xgboost", "structured_text_xgboost"]
KEYS = [(target, model) for target in TARGETS for model in MODEL_NAMES]
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
models, input_names, selected_parameters, tuning_rows = {}, {}, {}, []
lr_age_indices = [feature_names["lr_structured"].index(name) for name in ["num__age", "cat__gender_M"]]

def fit_lr(matrix, y, C=1.0, penalty="l2", max_iter=2000):
    model = LogisticRegression(C=C, penalty=penalty, solver="lbfgs", tol=1e-8,
        max_iter=max_iter, class_weight=None, fit_intercept=True)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        with threadpool_limits(limits=1):
            model.fit(matrix, y)
    return model

for target in TARGETS:
    ytr, yva = data["train"][target].to_numpy(), data["validation"][target].to_numpy()
    models[target, "dummy"] = DummyClassifier(strategy="prior").fit(matrices["lr_structured"]["train"], ytr)
    input_names[target, "dummy"] = "lr_structured"
    selected_parameters[target, "dummy"] = {"strategy": "training_prevalence"}
    models[target, "age_gender_lr"] = fit_lr(matrices["lr_structured"]["train"][:, lr_age_indices], ytr, penalty=None)
    input_names[target, "age_gender_lr"] = "lr_age"
    selected_parameters[target, "age_gender_lr"] = {"penalty": None, "features": ["num__age", "cat__gender_M"]}
    for name, matrix_name in [("structured_lr", "lr_structured"), ("structured_text_lr", "lr_text")]:
        best_ap, best_model, best_c = -np.inf, None, None
        for C in C_GRID:
            candidate = fit_lr(matrices[matrix_name]["train"], ytr, C=C, max_iter=20000 if matrix_name == "lr_text" else 2000)
            ap_value = average_precision_score(yva, candidate.predict_proba(matrices[matrix_name]["validation"])[:, 1])
            tuning_rows.append({"target": target, "model": name, "C": C, "validation_ap": ap_value})
            if ap_value > best_ap + 1e-12:
                best_ap, best_model, best_c = ap_value, candidate, C
            print(f"{target}/{name}: C={C:g}, validation AP={ap_value:.6f}", flush=True)
        models[target, name] = best_model
        input_names[target, name] = matrix_name
        selected_parameters[target, name] = {"C": best_c, "validation_ap": best_ap}

## Part 4 — three XGBoost models per outcome (Person 4, integrated by Person 5)

For each outcome, structured and text-inclusive models independently select from P4's nine candidates
using validation AP. Age/gender XGBoost uses the selected structured model's hyperparameters. Other
settings are P4's `hist`, `logloss`, seed 42, subsample 0.8 and colsample_bytree 0.8. Exact AP ties retain
the first candidate. The Stage ≥2 extension follows the delivered P4 settings and documented search recipe.

**Sparse zero correction:** the complete dense structured block is stored explicitly in CSR, including
zero and NaN entries, before adding the TF-IDF block. We do not call `eliminate_zeros`. The text block
is selected by feature names; previously reported 19/2,772/2,791 dimensions are not assumed.

In [ ]:
def dense_to_explicit_csr(matrix):
    matrix = np.asarray(matrix)
    rows, columns = matrix.shape
    return sp.csr_matrix((matrix.ravel(), np.tile(np.arange(columns, dtype=np.int32), rows),
        np.arange(0, (rows + 1) * columns, columns, dtype=np.int64)), shape=matrix.shape)

tree_age_indices = [feature_names["tree_structured"].index(name) for name in ["num__age", "cat__gender_M"]]
tree_text_indices = [i for i, name in enumerate(feature_names["tree_text"]) if name.startswith("text__")]
combined_tree_names = feature_names["tree_structured"] + [feature_names["tree_text"][i] for i in tree_text_indices]

def construct_model_inputs(part):
    values = {name: parts[part] for name, parts in matrices.items()}
    values["lr_age"] = values["lr_structured"][:, lr_age_indices]
    values["tree_age"] = values["tree_structured"][:, tree_age_indices]
    dense = np.asarray(values["tree_structured"])
    explicit = dense_to_explicit_csr(dense)
    assert explicit.nnz == dense.size
    assert np.array_equal(explicit.data, dense.ravel(), equal_nan=True)
    assert np.array_equal(explicit.toarray(), dense, equal_nan=True)
    text = sp.csr_matrix(values["tree_text"][:, tree_text_indices])
    values["tree_explicit_text"] = sp.hstack([explicit, text], format="csr")
    assert values["tree_explicit_text"].nnz == dense.size + text.nnz
    assert values["tree_explicit_text"].shape[1] == len(combined_tree_names)
    return values

model_inputs = {part: construct_model_inputs(part) for part in ("train", "validation")}
XGB_GRID = [{"n_estimators": n, "max_depth": depth, "learning_rate": rate,
             "subsample": 0.8, "colsample_bytree": 0.8}
    for depth in (2, 3, 4) for n, rate in ([(200, 0.05), (300, 0.05), (400, 0.03)]
        if depth != 4 else [(200, 0.05), (300, 0.03), (400, 0.03)])]

def fit_xgb(matrix, y, parameters):
    classifier = XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist",
        random_state=MODEL_SEED, n_jobs=N_JOBS, **parameters)
    return classifier.fit(matrix, y)

for target in TARGETS:
    ytr, yva = data["train"][target].to_numpy(), data["validation"][target].to_numpy()
    for name, matrix_name in [("structured_xgboost", "tree_structured"), ("structured_text_xgboost", "tree_explicit_text")]:
        best_ap, best_model, best_parameters = -np.inf, None, None
        for index, parameters in enumerate(XGB_GRID):
            candidate = fit_xgb(model_inputs["train"][matrix_name], ytr, parameters)
            ap_value = average_precision_score(yva, candidate.predict_proba(model_inputs["validation"][matrix_name])[:, 1])
            tuning_rows.append({"target": target, "model": name, "candidate": index, **parameters, "validation_ap": ap_value})
            if ap_value > best_ap:
                best_ap, best_model, best_parameters = ap_value, candidate, parameters.copy()
            print(f"{target}/{name}: candidate {index + 1}/{len(XGB_GRID)}, validation AP={ap_value:.6f}", flush=True)
        models[target, name] = best_model
        input_names[target, name] = matrix_name
        selected_parameters[target, name] = {**best_parameters, "validation_ap": best_ap}
    parameters = {key: value for key, value in selected_parameters[target, "structured_xgboost"].items() if key != "validation_ap"}
    models[target, "age_gender_xgboost"] = fit_xgb(model_inputs["train"]["tree_age"], ytr, parameters)
    input_names[target, "age_gender_xgboost"] = "tree_age"
    selected_parameters[target, "age_gender_xgboost"] = parameters
assert set(models) == set(KEYS)
save_csv(pd.DataFrame(tuning_rows), "validation_model_search.csv")

## Part 5 — validation freeze, test evaluation and sensitivity analyses (Person 5)

AP uses the step-wise average-precision definition, not a trapezoidal PR area. AUROC and Brier are
secondary metrics. Each model/outcome has its own largest validation threshold with Sensitivity ≥0.80;
predictions equal to the threshold are positive. Test data never select hyperparameters or thresholds.

Bootstrap samples patients with replacement within each patient's highest outcome category (none,
Stage 1 only, Stage ≥2), retaining all evaluated encounters for each selected patient. The **same draws**
are used for all 14 model/outcome combinations. There are 1,000 draws per scope; percentile 95% intervals
and paired AP/AUROC/Brier differences are reported. These intervals condition on the trained models,
fixed split and stratum sizes. No models are retrained inside Bootstrap; no multiplicity correction is applied.

Sensitivity subsets S1–S4 reuse the frozen models and thresholds: any vital observed, complete case,
exclude primary positives already positive on their first sample, and ≥2 current samples. S3 uses the
same primary-defined exclusion for both outcomes, matching P2/P5. S5–S6 are exported as descriptive
flags only. These subsets change the population and ascertainment; they are not causal experiments.

In [ ]:
thresholds, validation_probabilities, validation_rows = {}, {}, []
for target, name in KEYS:
    y = data["validation"][target].to_numpy()
    p = models[target, name].predict_proba(model_inputs["validation"][input_names[target, name]])[:, 1].astype(float)
    threshold = choose_threshold(y, p, SENSITIVITY_TARGET)
    metrics = PreparedMetrics(y, p, threshold).evaluate()
    reference = [average_precision_score(y, p), roc_auc_score(y, p), brier_score_loss(y, p)]
    assert np.allclose([metrics[k] for k in METRIC_NAMES[:3]], reference, rtol=0, atol=1e-12)
    assert metrics["sensitivity"] >= SENSITIVITY_TARGET - 1e-12
    assert (p[y == 1] >= np.nextafter(threshold, np.inf)).mean() < SENSITIVITY_TARGET
    thresholds[target, name], validation_probabilities[target, name] = threshold, p
    validation_rows.append({"target": target, "model": name, "threshold": threshold, **metrics})
validation_table = pd.DataFrame(validation_rows)
save_csv(validation_table, "validation_metrics_and_thresholds.csv")
freeze = {"created_at_utc": utc_now(), "sensitivity_target": SENSITIVITY_TARGET,
    "rule": "largest validation cutoff attaining sensitivity >= target; positive if probability >= cutoff",
    "selected_models": [{"target": t, "model": m, "parameters": selected_parameters[t, m], "threshold": thresholds[t, m]} for t, m in KEYS],
    "test_predictions_generated": False, "historical_test_results_previously_known": True,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
    "environment": ENVIRONMENT}
freeze_serialized = json.dumps(freeze, sort_keys=True, default=str)
freeze_hash = hashlib.sha256(freeze_serialized.encode()).hexdigest()
write_json(CHECKS / "analysis_freeze.json", freeze)
threshold_snapshot = thresholds.copy()
print("All 14 validation thresholds frozen before test inference.")
display(validation_table[["target", "model", "average_precision", "auroc", "threshold", "sensitivity", "ppv"]])

In [ ]:
write_json(CHECKS / "test_inference_start.json", {"started_at_utc": utc_now(), "analysis_freeze_sha256": freeze_hash})
for name, pre in fitted.items():
    matrices[name]["test"] = pre.transform(X["test"])
model_inputs["test"] = construct_model_inputs("test")
test_probabilities = {}
prediction_frames = []
for target in TARGETS:
    frame = data["test"][["stay_id", "subject_id", target]].rename(columns={target: "y_true"}).copy()
    frame.insert(0, "target", target)
    for name in MODEL_NAMES:
        p = models[target, name].predict_proba(model_inputs["test"][input_names[target, name]])[:, 1].astype(float)
        assert p.shape == (len(data["test"]),) and np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
        test_probabilities[target, name] = p
        frame[f"y_prob_{name}"] = p
    prediction_frames.append(frame)
save_csv(pd.concat(prediction_frames, ignore_index=True), "test_predictions_all_models.csv", PRIVATE)
assert thresholds == threshold_snapshot

In [ ]:
def patient_layout(labels):
    """Stratify patients by their highest outcome category among eligible encounters."""
    patients, inverse = np.unique(labels.subject_id.to_numpy(), return_inverse=True)
    encounter_stage = labels.aki_label.to_numpy(dtype=int) + labels.aki_stage2_label.to_numpy(dtype=int)
    patient_stage = np.zeros(len(patients), dtype=int)
    np.maximum.at(patient_stage, inverse, encounter_stage)
    groups = [np.flatnonzero(patient_stage == stage) for stage in [0, 1, 2]]
    return patients, inverse, patient_stage, groups

def draw_multiplicities(groups, n_patients, seed, scope_index, replicate):
    rng = np.random.default_rng(np.random.SeedSequence([seed, scope_index, replicate]))
    counts = np.zeros(n_patients, dtype=np.int64)
    for group in groups:
        if len(group):
            selected = group[rng.integers(len(group), size=len(group))]
            counts += np.bincount(selected, minlength=n_patients)
    return counts

PAIRS = [("structured_text_lr", "structured_lr"), ("structured_xgboost", "structured_lr"),
         ("structured_text_xgboost", "structured_xgboost"), ("structured_text_xgboost", "structured_text_lr")]
SUBSETS = ["S1_has_any_vital", "S2_complete_case", "S3_excl_positive_at_first_sample", "S4_two_or_more_samples"]

def evaluate_scope(labels, probabilities, scope, scope_index):
    assert len(labels) > 0, f"Empty sensitivity subset: {scope}"
    patients, inverse, stages, groups = patient_layout(labels)
    prepared = [PreparedMetrics(labels[t].to_numpy(), probabilities[t, m], thresholds[t, m]) for t, m in KEYS]
    points = [metric.evaluate() for metric in prepared]
    for (target, name), point in zip(KEYS, points):
        y, p = labels[target].to_numpy(), probabilities[target, name]
        assert abs(point["brier_score"] - brier_score_loss(y, p)) < 1e-12
        if np.unique(y).size == 2:
            assert abs(point["average_precision"] - average_precision_score(y, p)) < 1e-12
            assert abs(point["auroc"] - roc_auc_score(y, p)) < 1e-12
    samples = np.full((BOOTSTRAP_REPLICATES, len(KEYS), len(METRIC_NAMES)), np.nan)
    with threadpool_limits(limits=1):
        for replicate in range(BOOTSTRAP_REPLICATES):
            counts = draw_multiplicities(groups, len(patients), BOOTSTRAP_SEED, scope_index, replicate)
            assert counts.sum() == len(patients)
            weights = counts[inverse]
            for j, metric in enumerate(prepared):
                result = metric.evaluate(weights, include_calibration=scope == "test")
                samples[replicate, j] = [result[k] for k in METRIC_NAMES]
            if (replicate + 1) % 100 == 0 or replicate + 1 == BOOTSTRAP_REPLICATES:
                print(f"{scope}: paired patient Bootstrap {replicate + 1}/{BOOTSTRAP_REPLICATES}", flush=True)
    rows, intervals, pairs = [], [], []
    for j, (target, name) in enumerate(KEYS):
        base = {"scope": scope, "target": target, "model": name}
        rows.append({**base, "encounters": len(labels), "patients": len(patients),
            "positives": int(labels[target].sum()), "prevalence": float(labels[target].mean()),
            "threshold": thresholds[target, name], **points[j]})
        for k, metric in enumerate(METRIC_NAMES):
            if scope != "test" and metric in METRIC_NAMES[-3:]:
                continue
            values = samples[:, j, k]
            values = values[np.isfinite(values)]
            lo, hi = np.quantile(values, [0.025, 0.975]) if len(values) else (np.nan, np.nan)
            intervals.append({**base, "metric": metric, "estimate": points[j][metric],
                "ci_lower": lo, "ci_upper": hi, "valid_replicates": len(values), "requested_replicates": BOOTSTRAP_REPLICATES})
    for target in TARGETS:
        for a, b in PAIRS:
            ai, bi = KEYS.index((target, a)), KEYS.index((target, b))
            for k, metric in enumerate(METRIC_NAMES[:3]):
                values = samples[:, ai, k] - samples[:, bi, k]
                values = values[np.isfinite(values)]
                lo, hi = np.quantile(values, [0.025, 0.975]) if len(values) else (np.nan, np.nan)
                pairs.append({"scope": scope, "target": target, "model_a": a, "model_b": b,
                    "metric": metric, "difference_a_minus_b": points[ai][metric] - points[bi][metric],
                    "ci_lower": lo, "ci_upper": hi, "valid_replicates": len(values), "multiplicity_adjusted": False})
    # Fresh outputs only; these samples are never read to resume another run.
    np.save(CHECKS / f"bootstrap_{scope}.npy", samples)
    write_json(CHECKS / f"bootstrap_{scope}.json", {"patients": len(patients), "encounters": len(labels),
        "patient_strata": {str(s): len(g) for s, g in enumerate(groups)},
        "same_draws_for_all_models_and_targets": True, "method": "stratified patient cluster percentile Bootstrap",
        "calibration_ci": scope == "test", "keys": KEYS, "metrics": METRIC_NAMES})
    return pd.DataFrame(rows), pd.DataFrame(intervals), pd.DataFrame(pairs)

In [ ]:
test_labels = data["test"][["subject_id", "stay_id"] + TARGETS].copy()
test_flags = flags.set_index("stay_id").loc[test_labels.stay_id].reset_index()
assert np.array_equal(test_flags.subject_id, test_labels.subject_id)
point_frames, interval_frames, pair_frames = [], [], []
for scope_index, scope in enumerate(["test"] + SUBSETS):
    mask = np.ones(len(test_labels), dtype=bool) if scope == "test" else test_flags[scope].to_numpy(dtype=bool)
    points, intervals, pairs = evaluate_scope(test_labels.loc[mask].reset_index(drop=True),
        {key: probability[mask] for key, probability in test_probabilities.items()}, scope, scope_index)
    point_frames.append(points)
    interval_frames.append(intervals)
    pair_frames.append(pairs)
all_metrics = pd.concat(point_frames, ignore_index=True)
all_intervals = pd.concat(interval_frames, ignore_index=True)
all_pairs = pd.concat(pair_frames, ignore_index=True)
save_csv(all_metrics, "all_metrics.csv")
save_csv(all_intervals, "all_confidence_intervals.csv")
save_csv(all_pairs, "all_paired_differences.csv")
save_csv(all_metrics.loc[all_metrics.scope.eq("test")], "test_metrics.csv")
save_csv(all_metrics.loc[~all_metrics.scope.eq("test")], "sensitivity_metrics.csv")
summary_rows = []
for _, row in all_metrics.loc[all_metrics.scope.eq("test")].iterrows():
    entry = {key: row[key] for key in ["target", "model", "encounters", "positives", "prevalence", "threshold"]}
    for metric in ["average_precision", "auroc", "brier_score", "sensitivity", "specificity", "ppv", "npv"]:
        interval = all_intervals.loc[all_intervals.scope.eq("test") & all_intervals.target.eq(row.target)
            & all_intervals.model.eq(row.model) & all_intervals.metric.eq(metric)].iloc[0]
        entry[metric + " (95% CI)"] = f"{interval.estimate:.4f} [{interval.ci_lower:.4f}, {interval.ci_upper:.4f}]"
    summary_rows.append(entry)
comparison_table = pd.DataFrame(summary_rows)
save_csv(comparison_table, "primary_secondary_comparison.csv")
display(comparison_table)
display(all_pairs.loc[all_pairs.scope.eq("test")])

## Figures and interpretation checks

Calibration plots use up to ten quantile bins, keeping equal scores together, and retain bin counts.
Joint calibration intercept/slope and offset-only calibration-in-the-large are diagnostics; test predictions
are never recalibrated. A constant-score dummy has no identifiable joint slope/intercept. Intervals may
be undefined when a metric lacks the necessary outcome classes or predicted positives; valid draw counts
are retained. Feature importance and LR coefficients describe fitted associations, not causal effects.

In [ ]:
MODEL_LABELS = {"dummy": "Training-prevalence dummy", "age_gender_lr": "Age/gender LR",
    "structured_lr": "Structured LR", "structured_text_lr": "Structured + text LR",
    "age_gender_xgboost": "Age/gender XGBoost", "structured_xgboost": "Structured XGBoost",
    "structured_text_xgboost": "Structured + text XGBoost"}
TARGET_LABELS = {"aki_label": "Primary: Stage ≥1", "aki_stage2_label": "Secondary: Stage ≥2"}
COLORS = {name: plt.get_cmap("tab10")(i) for i, name in enumerate(MODEL_NAMES)}

def save_figure(fig, name):
    fig.savefig(FIGURES / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIGURES / f"{name}.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
for row, target in enumerate(TARGETS):
    y = test_labels[target].to_numpy()
    for name in MODEL_NAMES:
        p = test_probabilities[target, name]
        precision, recall, _ = precision_recall_curve(y, p)
        fpr, tpr, _ = roc_curve(y, p)
        axes[row, 0].step(recall, precision, where="post", label=MODEL_LABELS[name], color=COLORS[name])
        axes[row, 1].plot(fpr, tpr, label=MODEL_LABELS[name], color=COLORS[name])
    axes[row, 0].axhline(y.mean(), color="grey", linestyle="--", label="Test prevalence (reference)")
    axes[row, 1].plot([0, 1], [0, 1], color="grey", linestyle="--")
    for col, (xlabel, ylabel) in enumerate([("Recall", "Precision"), ("False positive rate", "Sensitivity")]):
        axes[row, col].set(xlabel=xlabel, ylabel=ylabel, xlim=(0, 1), ylim=(0, 1), title=TARGET_LABELS[target])
    axes[row, 0].legend(fontsize=8)
save_figure(fig, "01_test_pr_roc_both_outcomes")

calibration_rows = []
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, target in zip(axes, TARGETS):
    y = test_labels[target].to_numpy()
    for name in MODEL_NAMES:
        p = test_probabilities[target, name]
        bins = pd.qcut(p, q=10, labels=False, duplicates="drop") if np.ptp(p) > 0 else np.zeros(len(p))
        grouped = pd.DataFrame({"bin": bins, "probability": p, "y": y}).groupby("bin", dropna=False).agg(
            mean_probability=("probability", "mean"), observed_rate=("y", "mean"), count=("y", "size"), positives=("y", "sum")).reset_index()
        grouped.insert(0, "model", name)
        grouped.insert(0, "target", target)
        calibration_rows.append(grouped)
        ax.plot(grouped.mean_probability, grouped.observed_rate, "o-", markersize=3, color=COLORS[name], label=MODEL_LABELS[name])
    ax.plot([0, 1], [0, 1], "--", color="grey")
    relevant = pd.concat([g for g in calibration_rows if g.target.iloc[0] == target])
    upper = min(1.0, max(0.10, relevant[["mean_probability", "observed_rate"]].max().max() * 1.1))
    ax.set(xlabel="Mean predicted probability", ylabel="Observed positive proportion", title=TARGET_LABELS[target], xlim=(0, upper), ylim=(0, upper))
    ax.legend(fontsize=8)
save_csv(pd.concat(calibration_rows, ignore_index=True), "calibration_bins.csv")
save_figure(fig, "02_test_calibration_both_outcomes")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
flow_table = pd.DataFrame(flow)
ax.barh(np.arange(len(flow_table)), flow_table.encounters, color="#3977a8")
ax.set_yticks(np.arange(len(flow_table)), flow_table.step)
ax.invert_yaxis()
ax.set(xlabel="ED encounters", title="Cohort construction from fresh extraction")
for i, value in enumerate(flow_table.encounters):
    ax.text(value, i, f" {value:,}", va="center")
ax.set_xlim(0, flow_table.encounters.max() * 1.18)
save_figure(fig, "03_cohort_flow")

importance_rows = []
fig, axes = plt.subplots(2, 2, figsize=(15, 11), constrained_layout=True)
for row, target in enumerate(TARGETS):
    lr_coef = models[target, "structured_text_lr"].coef_[0]
    lr_order = np.argsort(np.abs(lr_coef))[-15:]
    axes[row, 0].barh([feature_names["lr_text"][i].replace("text__", "") for i in lr_order], lr_coef[lr_order])
    axes[row, 0].set(title=TARGET_LABELS[target] + ": LR coefficients", xlabel="Coefficient (mixed feature scales)")
    gain = models[target, "structured_text_xgboost"].get_booster().get_score(importance_type="gain")
    xgb_gain = np.array([gain.get(f"f{i}", 0.0) for i in range(len(combined_tree_names))])
    xgb_order = np.argsort(xgb_gain)[-15:]
    axes[row, 1].barh([combined_tree_names[i].replace("text__", "") for i in xgb_order], xgb_gain[xgb_order])
    axes[row, 1].set(title=TARGET_LABELS[target] + ": XGBoost gain", xlabel="Mean split gain")
    for feature, value in zip(feature_names["lr_text"], lr_coef):
        importance_rows.append({"target": target, "model": "structured_text_lr", "feature": feature, "measure": "coefficient", "value": value})
    for feature, value in zip(combined_tree_names, xgb_gain):
        importance_rows.append({"target": target, "model": "structured_text_xgboost", "feature": feature, "measure": "gain", "value": value})
save_csv(pd.DataFrame(importance_rows), "feature_associations.csv")
save_figure(fig, "04_feature_associations")

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, target in zip(axes, TARGETS):
    for name in ["structured_lr", "structured_text_lr", "structured_xgboost", "structured_text_xgboost"]:
        subset = all_intervals.loc[all_intervals.target.eq(target) & all_intervals.model.eq(name)
            & all_intervals.metric.eq("average_precision")].set_index("scope").loc[["test"] + SUBSETS]
        x = np.arange(len(subset))
        ax.plot(x, subset.estimate, "o-", label=MODEL_LABELS[name], color=COLORS[name])
        ax.vlines(x, subset.ci_lower, subset.ci_upper, color=COLORS[name], alpha=0.6)
    ax.set_xticks(range(5), ["All test", "S1", "S2", "S3", "S4"])
    ax.set(ylabel="AP (95% patient Bootstrap CI)", title=TARGET_LABELS[target])
    ax.legend(fontsize=8)
save_figure(fig, "05_sensitivity_ap")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for ax, target in zip(axes, TARGETS):
    pairs = all_pairs.loc[all_pairs.scope.eq("test") & all_pairs.target.eq(target) & all_pairs.metric.eq("average_precision")]
    y_positions = np.arange(len(pairs))
    ax.hlines(y_positions, pairs.ci_lower, pairs.ci_upper, color="#3977a8")
    ax.plot(pairs.difference_a_minus_b, y_positions, "o", color="#3977a8")
    ax.axvline(0, linestyle="--", color="grey")
    ax.set_yticks(y_positions, [MODEL_LABELS[r.model_a] + " minus\n" + MODEL_LABELS[r.model_b] for r in pairs.itertuples()])
    ax.set(xlabel="Paired difference in AP (95% CI)", title=TARGET_LABELS[target])
save_figure(fig, "06_paired_ap_differences")

In [ ]:
assert len(models) == 14 and thresholds == threshold_snapshot
assert len(all_metrics) == 14 * 5
assert len(all_pairs) == 2 * 4 * 3 * 5
assert all_intervals.requested_replicates.eq(BOOTSTRAP_REPLICATES).all()
assert len(query_audit) == 7 and not any(row["cache_hit"] for row in query_audit)
assert all((labels.aki_stage2_label <= labels.aki_label).all() for labels in data.values())
for target in TARGETS:
    assert np.unique(test_probabilities[target, "dummy"]).size == 1
    assert np.isclose(test_probabilities[target, "dummy"][0], data["train"][target].mean())
    dummy = all_metrics.loc[all_metrics.scope.eq("test") & all_metrics.target.eq(target) & all_metrics.model.eq("dummy")].iloc[0]
    assert np.isclose(dummy.average_precision, test_labels[target].mean()) and np.isclose(dummy.auroc, 0.5)
run_manifest = [{"file": path.relative_to(RUN_DIR).as_posix(), "bytes": path.stat().st_size,
    "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for path in RUN_DIR.rglob("*") if path.is_file()
    and path.name not in {"artifact_manifest.json", "run_status.json"}]
write_json(CHECKS / "artifact_manifest.json", run_manifest)
write_json(CHECKS / "run_status.json", {"status": "complete", "finished_at_utc": utc_now(),
    "models": 14, "outcomes": TARGETS, "bootstrap_replicates_per_scope": BOOTSTRAP_REPLICATES,
    "scopes": ["test"] + SUBSETS, "source": "fresh BigQuery extraction", "reads_existing_data_or_models": False,
    "fit_partition": "train", "threshold_partition": "validation", "analysis_freeze_sha256": freeze_hash,
    "historical_test_results_previously_known": True})
print("Complete:", RUN_DIR)
print("Comparison table: tables/primary_secondary_comparison.csv")
print("Paired differences and intervals: tables/all_paired_differences.csv")
print("Patient-level outputs are in private/; do not include them in a public submission.")

### Reading the outputs

- AP must be compared with the evaluated population's prevalence; the learned dummy uses **training** prevalence.
  Test prevalence is a descriptive reference, not a fitted prediction. Brier's oracle constant reference
  `π_test(1−π_test)` is not the dummy's actual score unless training and test prevalence coincide.
- A paired difference CI entirely above zero supports a positive AP/AUROC difference under this resampling
  procedure. For Brier, a negative difference favours model A. A CI crossing zero means uncertainty,
  not demonstrated equivalence. Comparisons are exploratory and unadjusted for multiplicity.
- AP across Stage ≥1 and Stage ≥2, or across sensitivity subsets, is affected by prevalence and case mix.
  Do not interpret a lower AP in the rarer endpoint as sufficient evidence of poorer discrimination.
- Validation sensitivity ≥0.80 does not guarantee test sensitivity ≥0.80. PPV also depends on prevalence
  and false positives. These models do not establish prospective clinical utility or transportability.
- Creatinine-based ascertainment, missing vitals, single-centre data and text reflecting existing clinical
  suspicion remain limitations. Early recognition at ED admission is not necessarily prediction before suspicion.

The notebook produces new CSV summaries and private predictions **as outputs only**. No generated CSV
is subsequently read into this run. Submit this notebook; keep row-level clinical data and Google credentials
in authorized private storage. The optional local builder/check scripts are maintenance tools, not runtime dependencies.